# CV-STQA on MAC-VQA: models M0 to M13, 20 random splits each, all 1,000 clips

One notebook that runs the whole study and writes a results package for a paper.

**What it does**
1. Reads all 1,000 clips and the six MOS labels (TSV, B, SR, S, U, Overall).
2. Extracts and caches the features once: ResNet-50 (layer 4 and layers 2+3), CLIP ViT-B/16 (or DINOv2-B), photometric descriptors, optical-flow descriptors and Video-Swin-T tokens.
3. Trains **14 models (M0 to M13) x 20 random splits = 280 runs**: random 720 train / 80 validation / 200 test, up to 40 epochs of 23 steps, early stopping on validation, best-validation epoch kept, test evaluated once per run. Training time and test-inference time are recorded for every run.
4. Computes SRCC, PLCC, KRCC and RMSE for all six scores per model per seed, with the 4-parameter logistic mapping of `nithincbabu7/non-lin-fit-python` fitted on **validation and applied fixed to test** (headline) and fitted on **test** (paper protocol).
5. Reports every model over **all 20 seeds** and over its **top 10 seeds** (ranked on validation), mean and median, with paired tests, DARNet comparison, hyper-parameter and logistic-parameter tables, 11 figures and a README.
6. Zips everything: `/kaggle/working/cvstqa_m13_results.zip`.

**Models**: M0 global mean | M1 +BiLSTM | M2 +temporal Transformer | M3 patch quality, average pooling | M4 +spatial attention | M5 +attribute temporal attention | M6 +consistency loss | **M7 M5+BGR graph (GCN)** | M8 M5+multi-stage ResNet | M9 +photometric | M10 +CLIP/DINOv2 | M11 +motion stream | M12 +query decoder | M13 +EMA, augmentation, robust loss, 3-member ensemble.

**Before you run (Kaggle)**: Accelerator = GPU (T4 or P100), **Internet = ON** (pretrained ResNet-50, CLIP, Video-Swin weights are downloaded once), attach the colonoscopy video dataset and the MOS dataset. Run the cells in order. Cell 8 is resumable: if the session stops, re-run Cells 1-6 and 8 and it continues from `runs.csv` (save a version of the notebook or download `cvstqa_m13_results/runs.csv` and put it back to keep progress across sessions).

**Time (estimate, not measured on your GPU)**: feature extraction about 20-40 min once; the 280 runs roughly 5-9 hours on a T4 (M13 trains 3 ensemble members per run). To check everything works first, set `SEEDS = [0, 1]` in Cell 1 (about 30-40 min), then restore `list(range(20))`.

**On the top-10 subset**: choosing the best 10 of 20 iterations is a selection step. This notebook ranks iterations by **validation** score, so test labels never choose the seeds; report the top-10 numbers together with the all-20 numbers.

In [ ]:
# ============ CELL 1 — imports & configuration ============
import os, gc, json, math, random, time, copy, re, warnings, shutil, zipfile, platform, itertools, sys
from pathlib import Path
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from scipy.stats import pearsonr, spearmanr, kendalltau, wilcoxon, ttest_rel, ttest_1samp, t as tdist
from scipy.optimize import curve_fit
import matplotlib
if "ipykernel" not in sys.modules: matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
cv2.setNumThreads(1)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU")
if DEVICE.type == "cpu":
    print("WARNING: no GPU. Turn on a GPU (Kaggle: Settings -> Accelerator) and Internet (needed once for the pretrained weights).")

# ---------------- paths ----------------
INPUT = Path(os.environ.get("CVSTQA_INPUT", "/kaggle/input"))
WORK  = Path(os.environ.get("CVSTQA_WORK", "/kaggle/working")); WORK.mkdir(parents=True, exist_ok=True)
CACHE = Path(os.environ.get("CVSTQA_CACHE", "/tmp/cvstqa_cache")); CACHE.mkdir(parents=True, exist_ok=True)   # big feature caches (~8 GB) live here, not in the zip
OUT   = WORK / "cvstqa_m13_results"
for sub in ["tables", "figures", "preds"]: (OUT / sub).mkdir(parents=True, exist_ok=True)
PRED_DIR, RUNS_CSV = OUT / "preds", OUT / "runs.csv"

# ---------------- experiment protocol ----------------
QUICK_TEST = os.environ.get("CVSTQA_SMOKE", "0") == "1"      # keep False for the real run
SEEDS      = list(range(20))          # 20 iterations = 20 random splits (seed s -> split s). Every model sees the same 20 splits.
TOP_K      = 10                       # report the best TOP_K of the 20 iterations per model (selected on VALIDATION, see Cell 9)
EPOCHS, BATCH = 40, 32                # 720 train clips / 32 = 23 steps (batches) per epoch
EARLY_STOP, PATIENCE = True, 10       # stop when validation Mean(6) SRCC has not improved for PATIENCE epochs; the best-validation epoch is always the one kept
TEST_FRAC, VAL_FRAC = 0.20, 0.10      # 200 test / 80 validation / 720 train out of 1000 clips
PREFERRED_MAPPING = "val"             # headline PLCC/RMSE use the logistic mapping fitted on VALIDATION and applied fixed to TEST
RUN_BRANCHES = False                  # True also runs M9b, M10b, M11b (single-feature branches of M5)
RANDOM_INIT = False                   # smoke-test only
PRETRAINED = True
if QUICK_TEST:
    SEEDS, TOP_K, EPOCHS, TEST_FRAC, VAL_FRAC = [0, 1, 2, 3], 2, 3, 0.30, 0.30
    PRETRAINED, RANDOM_INIT = False, True

# ---------------- fixed hyper-parameters (shared by all models unless a model is tuned in the optional Cell 8b) ----------------
HP = dict(D=256, DROPOUT=0.2, HEADS=4, SP_LAYERS=1, T_LAYERS=2,
          LR=5e-4, WD=1e-2, WARMUP=2, GRAD_CLIP=1.0,
          LAMBDA_RANK=0.1, RANK_MARGIN=0.05, LAMBDA_TEMP=0.5, TEMP_TAU=0.1)
HP_OVERRIDE = {}                      # per-model overrides, filled by the optional tuning cell
HP_SOURCE = "fixed defaults shared by every model (not tuned per model)"
# extras used only by the training-upgrade model M13
EMA_DECAY, TOKEN_DROP, FRAME_SWAP_P = 0.97, 0.10, 0.25
HUBER_DELTA, LAMBDA_PLCC, ATTR_W, ENS_K = 0.1, 0.1, [1.0, 1.5, 1.5, 1.0, 1.0, 1.0], 3
if QUICK_TEST: ENS_K = 2

# ---------------- data / feature sizes ----------------
NUM_FRAMES, IMAGE_SIZE, NUM_PATCHES = 16, 224, 49
CNN_DIM, MS_DIM, CLIP_DIM, SW_DIM, PH_DIM, MO_DIM = 2048, 1536, 768, 768, 16, 12
N_SEG, SEG_LEN, SEG_STRIDE = 8, 4, 2       # motion stream: 8 segments x 4 frames (every 2nd frame)
PATIENT_REGEX = r"^(\d+)-"
ATTRS = ["TSV", "B", "SR", "S", "U"]; TARGETS = ATTRS + ["O"]
SCORE_NAME = {"TSV": "TSV", "B": "B", "SR": "SR", "S": "S", "U": "U", "O": "Overall", "mean": "Mean(6)"}

# DARNet reference (paper Tables I and II; mean over its 10 trials). Order of tuple: SRCC, PLCC, KRCC, RMSE
DARNET = {"TSV": (.910, .924, .744, .252), "B": (.798, .821, .607, .459), "SR": (.841, .855, .654, .343),
          "S": (.928, .933, .768, .276), "U": (.906, .914, .737, .283), "O": (.940, .943, .790, .332)}
DARNET["mean"] = tuple(round(float(np.mean([DARNET[t][i] for t in TARGETS])), 3) for i in range(4))

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
print(f"{len(SEEDS)} seeds | {EPOCHS} epochs | batch {BATCH} | early stopping {EARLY_STOP} (patience {PATIENCE}) | quick-test {QUICK_TEST}")

## Cell 2 - locate the data (all 1,000 clips)

In [ ]:
# ============ CELL 2 — locate inputs & parse MOS ============
print("Top-level input folders:")
for p in sorted(INPUT.iterdir()):
    print("  ", p)
    for q in sorted(p.iterdir())[:6]:
        print("      ", q.name)

# optional: show what the split dataset contains (not used for training here)
for p in INPUT.rglob("*"):
    if p.is_dir() and "split" in p.name.lower():
        print("\nSplit-like folder:", p)
        for q in sorted(p.rglob("*"))[:20]:
            if q.is_file(): print("    ", q.relative_to(p), f"({q.stat().st_size/1e3:.1f} KB)")

mos_file = next(INPUT.rglob("MOS.txt"))
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}
all_videos = sorted(p for p in INPUT.rglob("*") if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
print(f"\nMOS.txt: {mos_file}\nVideos found: {len(all_videos)}")

vmap = {}
for p in all_videos:
    vmap.setdefault(p.name.lower(), str(p)); vmap.setdefault(p.stem.lower(), str(p))

mos_df = pd.read_csv(mos_file, sep=None, engine="python", header=None,
                     names=["video_id"] + TARGETS, skipinitialspace=True)
mos_df["video_id"] = mos_df["video_id"].astype(str).str.strip()
for c in TARGETS:
    mos_df[c] = pd.to_numeric(mos_df[c], errors="coerce")
mos_df = mos_df.dropna(subset=TARGETS).reset_index(drop=True)          # drops a header row if present
mos_df["video_path"] = mos_df["video_id"].str.lower().map(vmap)
if mos_df["video_path"].isna().any():
    mos_df["video_path"] = mos_df["video_id"].map(lambda s: vmap.get(Path(s).stem.lower()))
assert mos_df["video_path"].notna().all(), mos_df[mos_df["video_path"].isna()].head()
print("Labelled & matched clips:", len(mos_df))
print(mos_df[TARGETS].describe().loc[["min", "max", "mean"]].round(3))

mos_df["patient"] = mos_df["video_id"].map(lambda s: (re.match(PATIENT_REGEX, s) or [None, s])[1])
print("Unique patient/group prefixes:", mos_df["patient"].nunique(), "(expected ~40 exams — check!)")
Y = mos_df[TARGETS].values.astype(np.float32)          # [N, 6] raw MOS
N_VIDEOS = len(mos_df)
print(f"\nClips used: {N_VIDEOS} (the full dataset has 1,000)")
if N_VIDEOS != 1000 and not QUICK_TEST: print("WARNING: expected 1,000 labelled clips - check that both the video dataset and the MOS dataset are attached.")

## Cell 3 - feature extraction (one pass over the videos; cached; resumable)

In [ ]:
# ============ CELL 3 — extract & cache ALL features for ALL clips (one pass over the videos; resumable) ============
# l4  : ResNet-50 layer4 7x7 map            [N,16,49,2048]  (used by M0-M13)
# ms  : ResNet-50 layer2+layer3 pooled 7x7  [N,16,49,1536]  (multi-stage, low-level cues; M8+)
# clip: CLIP ViT-B/16 (or DINOv2-B) patches [N,16,49,768]   (M10+)
# ph  : photometric / sharpness descriptors [N,16,16]       (M9+)
# mo  : hand-crafted optical-flow descriptors [N,8,12]  and  sw: Video-Swin-T (Kinetics) tokens [N,8,768]  (M11+)
def photometric(fr):
    g = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
    hsv = cv2.cvtColor(fr, cv2.COLOR_RGB2HSV); S = hsv[..., 1].astype(np.float32) / 255.0; V = hsv[..., 2].astype(np.float32) / 255.0
    R, Gc, Bc = [fr[..., i].astype(np.float32) for i in range(3)]
    rg, yb = R - Gc, 0.5 * (R + Gc) - Bc
    colorful = np.sqrt(rg.std() ** 2 + yb.std() ** 2) + 0.3 * np.sqrt(rg.mean() ** 2 + yb.mean() ** 2)
    lap = cv2.Laplacian(g, cv2.CV_32F); gx = cv2.Sobel(g, cv2.CV_32F, 1, 0); gy = cv2.Sobel(g, cv2.CV_32F, 0, 1)
    edges = cv2.Canny((g * 255).astype(np.uint8), 50, 150).mean() / 255.0
    noise = np.median(np.abs(g - cv2.GaussianBlur(g, (3, 3), 0))) / 0.6745
    hist = np.histogram(g, bins=32, range=(0, 1))[0].astype(np.float32); hist /= hist.sum() + 1e-8
    p5, p95 = np.percentile(g, [5, 95])
    return np.array([g.mean(), g.std(), p5, p95, (g < 0.1).mean(), (g > 0.9).mean(), ((V > 0.94) & (S < 0.25)).mean(),
                     S.mean(), S.std(), colorful / 100.0, np.log1p(lap.var() * 1e3), np.sqrt(gx ** 2 + gy ** 2).mean(), edges, noise,
                     (R / (Gc + 1.0)).mean(), -(hist * np.log(hist + 1e-8)).sum()], np.float32)

def motion_feats(gray):                                   # list of SEG_LEN uint8 112x112 frames -> 12-d descriptor
    out = []
    for a, b in zip(gray[:-1], gray[1:]):
        flow = cv2.calcOpticalFlowFarneback(a, b, None, 0.5, 2, 15, 2, 5, 1.1, 0)
        fx, fy = flow[..., 0], flow[..., 1]; mag = np.sqrt(fx ** 2 + fy ** 2); mx, my = fx.mean(), fy.mean()
        res = np.sqrt((fx - mx) ** 2 + (fy - my) ** 2); div = np.gradient(fx, axis=1) + np.gradient(fy, axis=0)
        h = np.histogram(np.arctan2(fy, fx), bins=8, range=(-np.pi, np.pi), weights=mag)[0]; h = h / (h.sum() + 1e-8)
        diff = np.abs(a.astype(np.float32) - b.astype(np.float32)) / 255.0
        gx_, gy_ = np.meshgrid(np.arange(a.shape[1]), np.arange(a.shape[0]))
        back = cv2.remap(b, (gx_ + fx).astype(np.float32), (gy_ + fy).astype(np.float32), cv2.INTER_LINEAR)
        werr = np.abs(back.astype(np.float32) - a.astype(np.float32)).mean() / 255.0
        out.append([mag.mean(), mag.std(), np.percentile(mag, 90), mx, my, res.mean(), div.mean(), -(h * np.log(h + 1e-8)).sum(),
                    diff.mean(), (diff > 0.05).mean(), werr, mag.std() / (mag.mean() + 1e-6)])
    return np.mean(out, 0).astype(np.float32)

class VideoDS(Dataset):
    def __init__(self, paths, order, want_seg): self.paths, self.order, self.want_seg = paths, order, want_seg
    def __len__(self): return len(self.order)
    def _rd(self, cap, last):
        ok, fr = cap.read()
        if ok: return cv2.resize(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB), (IMAGE_SIZE, IMAGE_SIZE))
        return last if last is not None else np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), np.uint8)
    def __getitem__(self, k):
        i = int(self.order[k]); cap = cv2.VideoCapture(self.paths[i]); n = max(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), 1)
        frames, last = [], None
        for f in np.linspace(0, n - 1, NUM_FRAMES).astype(int):                                  # 16 uniformly sampled frames
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); last = self._rd(cap, last); frames.append(last)
        ph = np.stack([photometric(f) for f in frames])
        span = (SEG_LEN - 1) * SEG_STRIDE; starts = np.linspace(0, max(n - 1 - span, 0), N_SEG).astype(int)
        segs, mo = [], []
        for s in starts:                                                                          # 8 segments x 4 frames (stride 2)
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(s)); sf = []
            for q in range(SEG_LEN):
                last = self._rd(cap, sf[-1] if sf else last); sf.append(last)
                if q < SEG_LEN - 1:
                    for _ in range(SEG_STRIDE - 1): cap.grab()
            mo.append(motion_feats([cv2.resize(cv2.cvtColor(x, cv2.COLOR_RGB2GRAY), (112, 112)) for x in sf]))
            if self.want_seg: segs.extend(sf)
        cap.release()
        seg = torch.from_numpy(np.stack(segs)) if self.want_seg else torch.zeros(1, dtype=torch.uint8)
        return torch.from_numpy(np.stack(frames)), seg, torch.from_numpy(ph), torch.from_numpy(np.stack(mo)), i

# ---- backbones ----
def load_resnet():
    return models.resnet50(weights=models.ResNet50_Weights.DEFAULT if PRETRAINED else None).to(DEVICE).eval()
def resnet_stages(m, x):
    x = m.maxpool(m.relu(m.bn1(m.conv1(x)))); x = m.layer1(x); f2 = m.layer2(x); f3 = m.layer3(f2); f4 = m.layer4(f3); return f2, f3, f4

class _StubViT(nn.Module):        # smoke test only
    def __init__(s): super().__init__(); s.c = nn.Conv2d(3, 768, 16, 16)
    def forward(s, x): h = s.c(x).flatten(2).transpose(1, 2); return torch.cat([h[:, :1], h], 1)

def load_clip():
    """CLIP ViT-B/16 -> else DINOv2-B (Hugging Face; needs Internet ON once) -> else any CLIP/DINO folder attached under /kaggle/input. Returns (forward_fn, name, mean, std) or None."""
    if RANDOM_INIT:
        m = _StubViT().to(DEVICE).eval(); return (lambda x: m(x)), "stub", (0.5,) * 3, (0.5,) * 3
    cands = ["openai/clip-vit-base-patch16", "facebook/dinov2-base"]
    try:
        for cp in INPUT.rglob("config.json"):
            low = str(cp.parent).lower()
            if "clip" in low or "dino" in low: cands.append(str(cp.parent))
    except Exception: pass
    for c in cands:
        try:
            from transformers import CLIPVisionModel, AutoModel
            isclip = "clip" in c.lower()
            m = (CLIPVisionModel if isclip else AutoModel).from_pretrained(c).to(DEVICE).eval()
            mean, std = ((0.4815, 0.4578, 0.4082), (0.2686, 0.2613, 0.2758)) if isclip else ((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
            print("  loaded vision backbone:", c)
            return (lambda x, m=m: m(pixel_values=x).last_hidden_state), c, mean, std
        except Exception as e:
            print(f"  could not load {c}: {type(e).__name__}")
    return None

def load_swin():
    try:
        from torchvision.models.video import swin3d_t, Swin3D_T_Weights
        m = swin3d_t(weights=Swin3D_T_Weights.KINETICS400_V1 if PRETRAINED else None).to(DEVICE).eval(); return m
    except Exception as e:
        print("  could not load Video-Swin-T:", type(e).__name__); return None
def swin_tokens(m, x):            # x [b,3,32,224,224] -> [b,16,768]
    x = m.norm(m.features(m.pos_drop(m.patch_embed(x)))); return x.mean((2, 3))

t_load = time.time()
rn = load_resnet(); clip_bundle = load_clip(); swin = load_swin()
FEAT_AVAILABLE = ["l4", "ms", "ph", "mo"] + (["clip"] if clip_bundle else []) + (["sw"] if swin else [])
CLIP_NAME = clip_bundle[1] if clip_bundle else None
print("Features that will be available:", FEAT_AVAILABLE, "| CLIP-type backbone:", CLIP_NAME, "| Video-Swin:", swin is not None)
if not clip_bundle: print("!! No CLIP/DINOv2 backbone could be loaded: models M10-M13 will be SKIPPED. Turn Internet ON in the Kaggle settings (or attach a CLIP/DINOv2 weights dataset) and re-run this cell.")

SHAPES = {"l4": ((N_VIDEOS, NUM_FRAMES, NUM_PATCHES, CNN_DIM), np.float16), "ms": ((N_VIDEOS, NUM_FRAMES, NUM_PATCHES, MS_DIM), np.float16),
          "clip": ((N_VIDEOS, NUM_FRAMES, NUM_PATCHES, CLIP_DIM), np.float16), "sw": ((N_VIDEOS, N_SEG, SW_DIM), np.float16),
          "ph": ((N_VIDEOS, NUM_FRAMES, PH_DIM), np.float32), "mo": ((N_VIDEOS, N_SEG, MO_DIM), np.float32)}
need_gb = sum(np.prod(SHAPES[k][0]) * np.dtype(SHAPES[k][1]).itemsize for k in FEAT_AVAILABLE) / 1e9
print(f"Feature cache needs about {need_gb:.1f} GB in {CACHE} (free: {shutil.disk_usage(CACHE).free/1e9:.0f} GB)")

META, DONE = CACHE / "cache_meta.json", CACHE / "done.npy"
sig = {"feats": FEAT_AVAILABLE, "n": N_VIDEOS, "clip": CLIP_NAME, "random_init": RANDOM_INIT}
fresh = not (META.exists() and json.load(open(META)) == sig and DONE.exists() and all((CACHE / f"{k}.npy").exists() for k in FEAT_AVAILABLE))
mm = {k: np.lib.format.open_memmap(CACHE / f"{k}.npy", mode=("w+" if fresh else "r+"), dtype=SHAPES[k][1], shape=SHAPES[k][0]) for k in FEAT_AVAILABLE}
done = np.zeros(N_VIDEOS, bool) if fresh else np.load(DONE)
json.dump(sig, open(META, "w"))
todo = np.where(~done)[0]; print(f"Already cached: {int(done.sum())}/{N_VIDEOS} | to do: {len(todo)}")

EXTRACT_TIMES = {"resnet_s": 0.0, "clip_s": 0.0, "swin_s": 0.0, "decode_and_handcrafted_s": 0.0}
def _sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
if len(todo):
    imean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1); istd = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
    if clip_bundle: cmean = torch.tensor(clip_bundle[2], device=DEVICE).view(1, 3, 1, 1); cstd = torch.tensor(clip_bundle[3], device=DEVICE).view(1, 3, 1, 1)
    kmean = torch.tensor([0.43216, 0.394666, 0.37645], device=DEVICE).view(1, 3, 1, 1, 1); kstd = torch.tensor([0.22803, 0.22145, 0.216989], device=DEVICE).view(1, 3, 1, 1, 1)
    nw = 0 if QUICK_TEST else 2
    dl = DataLoader(VideoDS(mos_df["video_path"].tolist(), todo, swin is not None), batch_size=2, num_workers=nw, pin_memory=(DEVICE.type == "cuda"))
    t0 = time.time(); tw = time.time(); amp = dict(device_type=DEVICE.type, dtype=torch.float16, enabled=(DEVICE.type == "cuda"))
    def to_seq(t, b, d):       # [bT,d,7,7] -> [b,T,49,d]
        return t.flatten(2).transpose(1, 2).reshape(b, NUM_FRAMES, NUM_PATCHES, d)
    with torch.inference_mode():
        for bi, (fr, seg, ph, mo, ids) in enumerate(dl):
            EXTRACT_TIMES["decode_and_handcrafted_s"] += time.time() - tw
            b, ii = fr.shape[0], ids.numpy()
            x = fr.to(DEVICE).view(b * NUM_FRAMES, IMAGE_SIZE, IMAGE_SIZE, 3).permute(0, 3, 1, 2).float() / 255.0
            _sync(); t1 = time.time()
            with torch.autocast(**amp): f2, f3, f4 = resnet_stages(rn, (x - imean) / istd)
            mm["l4"][ii] = to_seq(f4.float(), b, CNN_DIM).half().cpu().numpy()
            mm["ms"][ii] = to_seq(torch.cat([F.adaptive_avg_pool2d(f2.float(), 7), F.adaptive_avg_pool2d(f3.float(), 7)], 1), b, MS_DIM).half().cpu().numpy()
            _sync(); EXTRACT_TIMES["resnet_s"] += time.time() - t1
            if clip_bundle:
                t1 = time.time()
                with torch.autocast(**amp): h = clip_bundle[0]((x - cmean) / cstd)
                n = int(round(math.sqrt(h.shape[1] - 1))); pt = h[:, 1:].float().reshape(b * NUM_FRAMES, n, n, -1).permute(0, 3, 1, 2)
                mm["clip"][ii] = to_seq(F.adaptive_avg_pool2d(pt, 7), b, CLIP_DIM).half().cpu().numpy()
                _sync(); EXTRACT_TIMES["clip_s"] += time.time() - t1
            if swin is not None:
                t1 = time.time()
                sx = (seg.to(DEVICE).permute(0, 4, 1, 2, 3).float() / 255.0 - kmean) / kstd
                with torch.autocast(**amp): st = swin_tokens(swin, sx)
                mm["sw"][ii] = st.float().view(b, N_SEG, 2, SW_DIM).mean(2).half().cpu().numpy()
                _sync(); EXTRACT_TIMES["swin_s"] += time.time() - t1
            mm["ph"][ii] = ph.numpy(); mm["mo"][ii] = mo.numpy(); done[ii] = True
            if bi % 25 == 0:
                for v in mm.values(): v.flush()
                np.save(DONE, done); print(f"  {int(done.sum())}/{N_VIDEOS} cached | {time.time()-t0:.0f}s")
            tw = time.time()
    for v in mm.values(): v.flush()
    np.save(DONE, done)
    EXTRACT_TIMES["total_s"] = time.time() - t0
    json.dump(EXTRACT_TIMES, open(OUT / "feature_extraction_times.json", "w"), indent=1)
if not len(todo) and (OUT / "feature_extraction_times.json").exists(): EXTRACT_TIMES = json.load(open(OUT / "feature_extraction_times.json"))
assert done.all(), "Some videos failed to cache"
del mm; gc.collect()
rn = clip_bundle = swin = None
if DEVICE.type == "cuda": torch.cuda.empty_cache()
print("Feature cache complete.", {k: round(v) for k, v in EXTRACT_TIMES.items()})

## Cell 4 - load the cached features

In [ ]:
# ============ CELL 4 — load the cached features (GPU if they fit) ============
STORE = {}
def _fits(nbytes):
    if DEVICE.type != "cuda": return False
    free = torch.cuda.mem_get_info()[0]; return nbytes < free - 5.5e9          # keep ~5.5 GB headroom for training
def load_big(name):
    arr = np.load(CACHE / f"{name}.npy", mmap_mode="r"); nb = arr.size * arr.dtype.itemsize
    if _fits(nb):
        t = torch.empty(arr.shape, dtype=torch.float16, device=DEVICE)
        for i in range(0, len(arr), 50): t[i:i + 50] = torch.from_numpy(np.ascontiguousarray(arr[i:i + 50])).to(DEVICE)
        where = "GPU"
    else:
        t = torch.from_numpy(np.ascontiguousarray(arr)); where = "CPU RAM"
    print(f"  {name:5s} {tuple(arr.shape)} {nb/1e9:.2f} GB -> {where}")
    return t
for k in [k for k in ["l4", "ms", "clip"] if k in FEAT_AVAILABLE]:
    STORE[f"p_{k}"] = load_big(k)
    # clip-level global per-frame feature = mean of the 49 patches (computed in chunks, kept in fp16 on the GPU)
    STORE[f"g_{k}"] = torch.cat([STORE[f"p_{k}"][i:i + 50].to(DEVICE).float().mean(2).half() for i in range(0, len(STORE[f"p_{k}"]), 50)])
STORE["ph"]   = torch.from_numpy(np.load(CACHE / "ph.npy")).to(DEVICE)
STORE["mo_h"] = torch.from_numpy(np.load(CACHE / "mo.npy")).to(DEVICE)
if "sw" in FEAT_AVAILABLE: STORE["mo_s"] = torch.from_numpy(np.load(CACHE / "sw.npy")).to(DEVICE)
Y = mos_df[TARGETS].values.astype(np.float32)
print("loaded:", {k: tuple(v.shape) for k, v in STORE.items()})

def take(name, idx):
    t = STORE[name]; return t[idx.to(t.device)].to(DEVICE, non_blocking=True).float()

## Cell 5 - models M0 to M13

In [ ]:
# ============ CELL 5 — the 14 models M0-M13 (one clean code path, controlled by flags) ============
BASE = dict(use_patches=False, temporal="none", spatial_pool=None, temporal_pool=None, temp_loss=False, global_resid=False,
            ms=False, clip=False, ph=False, mo=False, bgr=False, qdec=False, ema=False, aug=False, robust=False, ens=1)
def V(parent=None, **kw):
    d = dict(BASE if parent is None else parent); d.update(kw); return d
M5 = V(use_patches=True, temporal="transformer", spatial_pool="attn", temporal_pool="attn", global_resid=True)
VARIANTS = {
    "M0":  V(),                                                                       # mean-pooled global ResNet-50 features
    "M1":  V(temporal="lstm"),                                                        # + BiLSTM over time
    "M2":  V(temporal="transformer"),                                                 # + temporal Transformer on global features
    "M3":  V(use_patches=True, temporal="transformer", spatial_pool="avg", temporal_pool="mean", global_resid=True),   # patch-level quality, average pooling
    "M4":  V(use_patches=True, temporal="transformer", spatial_pool="attn", temporal_pool="mean", global_resid=True),  # + spatial criticality (alpha)
    "M5":  M5,                                                                        # + attribute-specific temporal criticality (beta)
    "M6":  V(M5, temp_loss=True),                                                     # + temporal-consistency loss
    "M7":  V(M5, bgr=True),                                                           # M5 + BGR graph (GCN over the 5 attributes)
    "M8":  V(M5, ms=True),                                                            # M5 + multi-stage ResNet features
    "M9":  V(M5, ms=True, ph=True),                                                   # + photometric / sharpness descriptors
    "M10": V(M5, ms=True, ph=True, clip=True),                                       # + CLIP/DINOv2 patch stream
    "M11": V(M5, ms=True, ph=True, clip=True, mo=True),                              # + motion stream (optical flow + Video-Swin)
    "M12": V(M5, ms=True, ph=True, clip=True, mo=True, qdec=True),                   # + query decoder with attribute-to-attribute attention
    "M13": V(M5, ms=True, ph=True, clip=True, mo=True, qdec=True, ema=True, aug=True, robust=True, ens=ENS_K),   # + EMA, augmentation, robust loss, ensemble
}
BRANCHES = {"M9b": V(M5, ph=True), "M10b": V(M5, clip=True), "M11b": V(M5, mo=True)}          # single-feature branches of M5 (optional)
VARIANTS.update(BRANCHES)
DESCRIPTION = {
    "M0": "global ResNet-50 features, mean over time", "M1": "M0 + BiLSTM", "M2": "M0 + temporal Transformer",
    "M3": "patch quality + average pooling", "M4": "M3 + spatial attention (alpha)", "M5": "M4 + attribute temporal attention (beta)",
    "M6": "M5 + temporal-consistency loss", "M7": "M5 + BGR graph (GCN)", "M8": "M5 + multi-stage ResNet (L2+L3+L4)",
    "M9": "M8 + photometric descriptors", "M10": "M9 + CLIP/DINOv2 patch stream", "M11": "M10 + motion stream (flow + Video-Swin)",
    "M12": "M11 + query decoder (attribute-to-attribute attention)", "M13": f"M12 + EMA + augmentation + robust loss + {ENS_K}-member ensemble",
    "M9b": "M5 + photometric only", "M10b": "M5 + CLIP/DINOv2 only", "M11b": "M5 + motion only"}
ALL_MODELS = [f"M{i}" for i in range(14)] + (list(BRANCHES) if RUN_BRANCHES else [])
def needs(cfg):
    n = set()
    if cfg["ms"]: n.add("ms")
    if cfg["clip"]: n.add("clip")
    return n
def available(name):
    cfg = VARIANTS[name]
    return needs(cfg) <= set(FEAT_AVAILABLE)
RUN_MODELS = [m for m in ALL_MODELS if available(m)]
skipped = [m for m in ALL_MODELS if m not in RUN_MODELS]
print("Models to run:", RUN_MODELS, "| skipped (missing features):", skipped)

def hp_for(name): return {**HP, **HP_OVERRIDE.get(name, {})}
def mlp(i, h, o, p): return nn.Sequential(nn.Linear(i, h), nn.GELU(), nn.Dropout(p), nn.Linear(h, o))
def tf_encoder(d, layers, p, heads):
    return nn.TransformerEncoder(nn.TransformerEncoderLayer(d, heads, 2 * d, p, activation="gelu", batch_first=True, norm_first=True), layers, enable_nested_tensor=False)

class BGR(nn.Module):
    """Graph reasoning over the 5 attribute nodes (DARNet-style). Edges: |PLCC| > 0.55 between attributes in the TRAINING labels (+ self loops).
    2-layer GCN; residual with a learnable gate gamma (starts at 0), so the module can only help once it has learned something."""
    def __init__(self, d, adj, dn=64, p=0.2):
        super().__init__()
        self.register_buffer("A", adj); self.inp = nn.Linear(1, dn); self.zp = nn.Linear(d, dn); self.emb = nn.Parameter(torch.zeros(1, 5, dn))
        self.l1, self.l2 = nn.Linear(dn, dn), nn.Linear(dn, dn); self.drop = nn.Dropout(p)
        self.out, self.gout = nn.Linear(dn, 1), nn.Linear(dn, d); self.gamma = nn.Parameter(torch.zeros(1))
    def forward(self, ya, zg):
        h = F.gelu(self.inp(ya.unsqueeze(-1)) + self.emb + self.zp(zg).unsqueeze(1))
        h = self.drop(F.gelu(self.A @ self.l1(h))); h = self.A @ self.l2(h)
        return ya + self.gamma * self.out(h).squeeze(-1), self.gamma * self.gout(h.mean(1))

class Net(nn.Module):
    def __init__(self, cfg, hp, norm=None, adj=None):
        super().__init__()
        d, p, H = hp["D"], hp["DROPOUT"], hp["HEADS"]; self.cfg, self.hp, self.d = cfg, hp, d
        T, N = NUM_FRAMES, NUM_PATCHES
        self.has_sw = "sw" in FEAT_AVAILABLE
        for k, n in [("ph_mu", PH_DIM), ("ph_sd", PH_DIM), ("mh_mu", MO_DIM), ("mh_sd", MO_DIM), ("ms_mu", SW_DIM), ("ms_sd", SW_DIM)]:
            self.register_buffer(k, torch.zeros(n) if k.endswith("mu") else torch.ones(n))
        if norm is not None:
            for k, v in norm.items(): getattr(self, k).copy_(torch.as_tensor(v))
        gdim = CNN_DIM + (MS_DIM if cfg["ms"] else 0)
        self.gproj = nn.Sequential(nn.Dropout(p), nn.Linear(gdim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p))
        if cfg["clip"]: self.cgproj = nn.Sequential(nn.Dropout(p), nn.Linear(CLIP_DIM, d), nn.LayerNorm(d))
        if cfg["ph"]: self.phf, self.phc = mlp(PH_DIM, d, d, p), mlp(2 * PH_DIM, d, d, p)
        if cfg["mo"]:
            self.mproj = nn.Sequential(nn.Linear(MO_DIM + (SW_DIM if self.has_sw else 0), d), nn.LayerNorm(d), nn.GELU())
            self.mpos = nn.Parameter(torch.zeros(1, N_SEG, d)); nn.init.trunc_normal_(self.mpos, std=0.02)
            self.menc, self.mfuse = tf_encoder(d, 1, p, H), nn.Linear(d, d)
        if cfg["temporal"] == "transformer":
            self.tpos = nn.Parameter(torch.zeros(1, T, d)); nn.init.trunc_normal_(self.tpos, std=0.02)
            self.tenc, self.tnorm = tf_encoder(d, hp["T_LAYERS"], p, H), nn.LayerNorm(d)
        elif cfg["temporal"] == "lstm":
            self.lstm = nn.LSTM(d, d // 2, batch_first=True, bidirectional=True)
        if not cfg["use_patches"]:
            self.head = mlp(d, d, len(TARGETS), p); return
        pdim = CNN_DIM + (MS_DIM if cfg["ms"] else 0)
        self.pproj = nn.Sequential(nn.Dropout(p), nn.Linear(pdim, d), nn.LayerNorm(d), nn.GELU(), nn.Dropout(p))
        if cfg["clip"]:
            self.cpproj = nn.Sequential(nn.Dropout(p), nn.Linear(CLIP_DIM, d), nn.LayerNorm(d)); self.cgate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        self.ppos = nn.Parameter(torch.zeros(1, N, d)); nn.init.trunc_normal_(self.ppos, std=0.02)
        self.sp_enc = tf_encoder(d, hp["SP_LAYERS"], p, H)
        self.gate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        if cfg["qdec"]:
            self.queries = nn.Parameter(torch.randn(1, 6, d) * 0.02); self.qtpos = nn.Parameter(torch.zeros(1, T, 1, d)); nn.init.trunc_normal_(self.qtpos, std=0.02)
            self.dec = nn.TransformerDecoder(nn.TransformerDecoderLayer(d, H, 2 * d, p, activation="gelu", batch_first=True, norm_first=True), 2)
            self.dnorm = nn.LayerNorm(d); self.qa = nn.ModuleList([mlp(d, d // 2, 1, p) for _ in ATTRS])
        else:
            self.qh = nn.ModuleList([mlp(d, d // 2, 1, p) for _ in ATTRS])                      # local quality per attribute
            if cfg["spatial_pool"] == "attn": self.sh = nn.ModuleList([mlp(d, d // 2, 1, p) for _ in ATTRS])
            self.fuse = nn.Sequential(nn.Linear(d + len(ATTRS), d), nn.LayerNorm(d), nn.GELU())
            if cfg["temporal_pool"] == "attn": self.th = nn.ModuleList([mlp(d, d // 4, 1, p) for _ in ATTRS])
        if cfg["global_resid"]:
            self.gres = nn.Linear(d, len(ATTRS)); nn.init.zeros_(self.gres.weight); nn.init.zeros_(self.gres.bias)
        if cfg["bgr"]: self.bgr = BGR(d, adj if adj is not None else torch.eye(5), p=p)
        self.oh = mlp(len(ATTRS) + d + (d if (cfg["bgr"] or cfg["qdec"]) else 0), d, 1, p)

    def temporal(self, x):
        t = self.cfg["temporal"]
        if t == "transformer": return self.tnorm(self.tenc(x + self.tpos))
        if t == "lstm": return self.lstm(x)[0]
        return x

    def frame_context(self, B):
        """frame-level global feature g [B,T,d] + clip-level photometric / motion context."""
        G = B["g_l4"] if not self.cfg["ms"] else torch.cat([B["g_l4"], B["g_ms"]], -1)
        g = self.gproj(G); pc = mctx = None
        if self.cfg["clip"]: g = g + self.cgproj(B["g_clip"])
        if self.cfg["ph"]:
            ph = (B["ph"] - self.ph_mu) / self.ph_sd; g = g + self.phf(ph); pc = self.phc(torch.cat([ph.mean(1), ph.std(1)], -1))
        if self.cfg["mo"]:
            mi = (B["mo_h"] - self.mh_mu) / self.mh_sd
            if self.has_sw: mi = torch.cat([mi, (B["mo_s"] - self.ms_mu) / self.ms_sd], -1)
            mctx = self.mfuse(self.menc(self.mproj(mi) + self.mpos).mean(1))
        return g, pc, mctx

    def forward(self, B):
        cfg = self.cfg; g, pc, mctx = self.frame_context(B)
        if not cfg["use_patches"]:
            return dict(pred=self.head(self.temporal(g).mean(1)), Q=None, alpha=None, beta=None, G=B["g_l4"])
        b, T = g.shape[:2]; N = NUM_PATCHES
        P = B["p_l4"] if not cfg["ms"] else torch.cat([B["p_l4"], B["p_ms"]], -1)
        if self.training and cfg["aug"] and TOKEN_DROP > 0: P = P * (torch.rand(b, T, N, 1, device=P.device) > TOKEN_DROP).float()
        x = self.pproj(P.reshape(b * T, N, -1))
        if cfg["clip"]:
            xc = self.cpproj(B["p_clip"].reshape(b * T, N, CLIP_DIM)); x = x + self.cgate(torch.cat([x, xc], -1)) * xc
        c = self.sp_enc(x + self.ppos); gm = self.gate(torch.cat([x, c], -1)); f = gm * x + (1 - gm) * c      # adaptive CNN/Transformer fusion
        Q = alpha = beta = None
        if cfg["qdec"]:
            z = self.temporal(g)
            if mctx is not None: z = z + mctx.unsqueeze(1)
            zg = z.mean(1) + (pc if pc is not None else 0)
            tok = (f.view(b, T, N, -1) + self.qtpos).reshape(b, T * N, -1)
            qo = self.dnorm(self.dec(self.queries.expand(b, -1, -1), tok))                                       # [b,6,d]
            ya = torch.cat([h(qo[:, i]) for i, h in enumerate(self.qa)], -1)
            if cfg["global_resid"]: ya = ya + self.gres(zg)
            extra = qo[:, 5]
        else:
            q = torch.sigmoid(torch.cat([h(f) for h in self.qh], -1))                                            # [bT,N,5] local quality
            alpha = torch.full_like(q, 1.0 / N)
            if cfg["spatial_pool"] == "attn":
                alpha = torch.cat([h(f) for h in self.sh], -1).softmax(1); Qf = (alpha * q).sum(1)
            else:
                Qf = q.mean(1)
            Q = Qf.view(b, T, len(ATTRS))                                                                         # frame-level quality trajectories
            z = self.temporal(self.fuse(torch.cat([g, Q], -1)))
            if mctx is not None: z = z + mctx.unsqueeze(1)
            beta = torch.full_like(Q, 1.0 / T)
            if cfg["temporal_pool"] == "attn":
                beta = torch.cat([h(z) for h in self.th], -1).softmax(1); ya = (beta * Q).sum(1)
            else:
                ya = Q.mean(1)
            zg = z.mean(1) + (pc if pc is not None else 0)
            if cfg["global_resid"]: ya = ya + self.gres(zg)
            alpha = alpha.view(b, T, N, len(ATTRS)); extra = None
        if cfg["bgr"]: ya, extra = self.bgr(ya, zg)
        yo = self.oh(torch.cat([ya, zg] + ([extra] if extra is not None else []), -1))
        return dict(pred=torch.cat([ya, yo], -1), Q=Q, alpha=alpha, beta=beta, G=B["g_l4"])

def count_params(name):
    cfg, hp = VARIANTS[name], hp_for(name)
    return sum(p.numel() for p in Net(cfg, hp, adj=torch.eye(5)).parameters() if p.requires_grad)
PARAMS = {m: count_params(m) for m in RUN_MODELS}
for m in RUN_MODELS: print(f"{m:4s} {PARAMS[m]:>10,} trainable parameters" + (f"  (x{VARIANTS[m]['ens']} ensemble members)" if VARIANTS[m]["ens"] > 1 else "") + f"  | {DESCRIPTION[m]}")

## Cell 6 - losses, logistic mapping (reference-repo function), metrics, training and timing

In [ ]:
# ============ CELL 6 — batches, losses, logistic mapping, metrics, training, timing ============
def get_batch(idx, cfg):
    idx = torch.as_tensor(idx); B = {"g_l4": take("g_l4", idx)}
    if cfg["ms"]: B["g_ms"] = take("g_ms", idx)
    if cfg["clip"]: B["g_clip"] = take("g_clip", idx)
    if cfg["use_patches"]:
        B["p_l4"] = take("p_l4", idx)
        if cfg["ms"]: B["p_ms"] = take("p_ms", idx)
        if cfg["clip"]: B["p_clip"] = take("p_clip", idx)
    if cfg["ph"]: B["ph"] = take("ph", idx)
    if cfg["mo"]:
        B["mo_h"] = take("mo_h", idx)
        if "mo_s" in STORE: B["mo_s"] = take("mo_s", idx)
    return B

def aug_batch(B):
    """training-time augmentation for M13: with prob FRAME_SWAP_P a frame is replaced by its temporal neighbour (all frame-wise streams, consistently)."""
    b, T = B["g_l4"].shape[:2]; dev = B["g_l4"].device
    shift = torch.randint(0, 2, (b, T), device=dev) * 2 - 1
    swap = torch.rand(b, T, device=dev) < FRAME_SWAP_P
    src = (torch.arange(T, device=dev).unsqueeze(0) + shift * swap).clamp(0, T - 1)
    rows = torch.arange(b, device=dev).unsqueeze(1)
    return {k: (v[rows, src] if k in ("g_l4", "g_ms", "g_clip", "p_l4", "p_ms", "p_clip", "ph") else v) for k, v in B.items()}

def compute_losses(out, y, cfg, hp):
    pred = out["pred"]; w = torch.tensor(ATTR_W, device=pred.device)
    if cfg["robust"]: reg = (F.smooth_l1_loss(pred, y, beta=HUBER_DELTA, reduction="none") * w).mean()
    else: reg = F.mse_loss(pred, y)
    pdiff = pred[:, None] - pred[None]; tdiff = y[:, None] - y[None]; valid = (tdiff > 0.05).float()
    rank = (valid * F.relu(hp["RANK_MARGIN"] - pdiff)).sum() / (valid.sum() + 1e-6)
    loss = reg + hp["LAMBDA_RANK"] * rank
    if cfg["robust"]:                                                    # differentiable PLCC term
        pc = pred - pred.mean(0); yc = y - y.mean(0)
        plcc = (pc * yc).sum(0) / (pc.norm(dim=0) * yc.norm(dim=0) + 1e-6)
        loss = loss + LAMBDA_PLCC * (1 - plcc).mean()
    if cfg["temp_loss"] and out["Q"] is not None:
        Gn = F.normalize(out["G"], dim=-1); dist = 1 - (Gn[:, 1:] * Gn[:, :-1]).sum(-1)
        wt = torch.exp(-dist / hp["TEMP_TAU"]).detach()
        loss = loss + hp["LAMBDA_TEMP"] * (wt * (out["Q"][:, 1:] - out["Q"][:, :-1]).pow(2).sum(-1)).mean()
    return loss

# ---------------- non-linear logistic mapping: the 4-parameter logistic of nithincbabu7/non-lin-fit-python (analysis.py), same call pattern ----------------
class NonLinearFit:
    """Re-implementation of NonLinearFit.four_param_logistic_QA from the reference repo:
       f(y) = (b1 - b2) / (1 + exp((b3 - y) / |b4|)) + b2,  fitted with scipy curve_fit(f, y=prediction, x=MOS), maxfev=10000.
       (only change: the exponent is clipped to +-60 so that np.exp cannot overflow.)"""
    def __init__(self): self.maxfev = 10000; self.params = None
    @staticmethod
    def four_param_logistic_QA(x, beta1, beta2, beta3, beta4):
        return ((beta1 - beta2) / (1 + np.exp(np.clip((beta3 - x) / (np.abs(beta4) + 1e-12), -60, 60)))) + beta2
    def fit(self, x, y, beta_init=None):          # x = target MOS, y = model prediction (same argument order as the reference code)
        if beta_init is None: beta_init = [np.min(x), np.max(x), np.mean(y), 1]
        self.params, _ = curve_fit(self.four_param_logistic_QA, y, x, p0=beta_init, maxfev=self.maxfev)
    def transform(self, y): return self.four_param_logistic_QA(y, *self.params)

def fit_mapping(mos, pred):
    """Fit the mapping on (pred -> mos). Tries the reference initial guess [min(MOS), max(MOS), mean(pred), 1] and, as a fallback, an increasing-curve guess;
    keeps the better fit, and only if it beats the identity mapping on the data it was fitted on. Returns (fit or None, ok, which_init)."""
    best = None
    for k, b0 in enumerate([[mos.min(), mos.max(), pred.mean(), 1.0], [mos.max(), mos.min(), pred.mean(), pred.std() / 4 + 1e-3]]):
        try:
            f = NonLinearFit(); f.fit(mos, pred, b0); err = np.mean((f.transform(pred) - mos) ** 2)
            if np.isfinite(err) and (best is None or err < best[0]): best = (err, f, k)
        except Exception: pass
    if best is None or best[0] > np.mean((pred - mos) ** 2): return None, False, -1
    return best[1], True, best[2]
def apply_mapping(f, pred): return pred if f is None else f.transform(pred)

def _corr(fn, a, b): return 0.0 if np.std(a) < 1e-9 or np.std(b) < 1e-9 else float(fn(a, b)[0])
def _rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))
def mean_srcc(y, p): return float(np.mean([_corr(spearmanr, y[:, j], p[:, j]) for j in range(y.shape[1])]))
METRICS = ["srcc", "krcc", "plcc_raw", "rmse_raw", "plcc_val", "rmse_val", "plcc_test", "rmse_test"]
def all_metrics(y_te, p_te, y_va, p_va):
    """per score: SRCC, KRCC, PLCC/RMSE raw, with the mapping fitted on VALIDATION (applied fixed to test) and fitted on TEST; the 4 logistic parameters of each fit; plus the six-score mean."""
    r = {}
    for j, t in enumerate(TARGETS):
        a, b = y_te[:, j], p_te[:, j]
        r[f"{t}_srcc"], r[f"{t}_krcc"] = _corr(spearmanr, a, b), _corr(kendalltau, a, b)
        r[f"{t}_plcc_raw"], r[f"{t}_rmse_raw"] = _corr(pearsonr, a, b), _rmse(a, b)
        for tag, (fx, fy) in {"val": (y_va[:, j], p_va[:, j]), "test": (a, b)}.items():
            f, ok, k = fit_mapping(fx, fy); m = apply_mapping(f, b)
            r[f"{t}_plcc_{tag}"], r[f"{t}_rmse_{tag}"], r[f"{t}_fitok_{tag}"], r[f"{t}_init_{tag}"] = _corr(pearsonr, a, m), _rmse(a, m), ok, k
            for q in range(4): r[f"{t}_{tag[0]}b{q+1}"] = float(f.params[q]) if f is not None else np.nan      # vb1..vb4 (val-fit), tb1..tb4 (test-fit)
    for m in METRICS: r[f"mean_{m}"] = float(np.mean([r[f"{t}_{m}"] for t in TARGETS]))
    return r

# ---------------- training ----------------
def _sync():
    if DEVICE.type == "cuda": torch.cuda.synchronize()
@torch.no_grad()
def predict(model, idx, lo, hi, cfg, bs=64):
    model.eval(); ps = []
    for i in range(0, len(idx), bs): ps.append(model(get_batch(torch.as_tensor(idx[i:i + bs]), cfg))["pred"].float().cpu())
    return torch.cat(ps).numpy() * (hi - lo) + lo                 # back to the original MOS scale

class EMA:
    def __init__(self, model, decay): self.model = copy.deepcopy(model).eval(); self.decay = decay
    @torch.no_grad()
    def update(self, model):
        for pe, p in zip(self.model.parameters(), model.parameters()): pe.mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)

def make_norm(tr):
    tr_t = torch.as_tensor(tr); n = {}
    ph = STORE["ph"][tr_t.to(STORE["ph"].device)].reshape(-1, PH_DIM); n["ph_mu"], n["ph_sd"] = ph.mean(0), ph.std(0) + 1e-6
    mh = STORE["mo_h"][tr_t.to(STORE["mo_h"].device)].reshape(-1, MO_DIM); n["mh_mu"], n["mh_sd"] = mh.mean(0), mh.std(0) + 1e-6
    if "mo_s" in STORE:
        ms = STORE["mo_s"][tr_t.to(STORE["mo_s"].device)].reshape(-1, SW_DIM).float(); n["ms_mu"], n["ms_sd"] = ms.mean(0), ms.std(0) + 1e-6
    return {k: v.cpu() for k, v in n.items()}
def make_adj(ytr):
    c = np.corrcoef(ytr[:, :5].T); A = ((np.abs(c) > 0.55) | np.eye(5, dtype=bool)).astype(np.float32); dinv = 1 / np.sqrt(A.sum(1))
    return torch.from_numpy(A * dinv[:, None] * dinv[None])

def train_member(cfg, hp, tr, va, member_seed, lo, hi, Ys, norm, adj):
    seed_everything(member_seed)
    model = Net(cfg, hp, norm, adj).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["LR"], weight_decay=hp["WD"]); W = hp["WARMUP"]
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda e: (e + 1) / W if e < W else max(0.02, 0.5 * (1 + math.cos(math.pi * (e - W) / max(1, EPOCHS - W)))))
    ema = EMA(model, EMA_DECAY) if cfg["ema"] else None
    best, best_state, best_ep, best_step, bad, steps, steps_ep1, ep = -9, None, 0, 0, 0, 0, 0, 0
    tr_t = torch.as_tensor(tr)
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_t[torch.randperm(len(tr_t))]; nb = 0
        for i in range(0, len(perm), BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 4: continue
            B = get_batch(idx, cfg)
            if cfg["aug"]: B = aug_batch(B)
            loss = compute_losses(model(B), Ys[idx.to(Ys.device)], cfg, hp)
            opt.zero_grad(set_to_none=True); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), hp["GRAD_CLIP"]); opt.step()
            if ema is not None: ema.update(model)
            nb += 1; steps += 1
        if ep == 1: steps_ep1 = nb
        sched.step()
        evalm = ema.model if ema is not None else model
        vs = mean_srcc(Y[va], predict(evalm, va, lo, hi, cfg))
        if vs > best + 1e-4:
            best, best_ep, best_step, bad = vs, ep, steps, 0
            best_state = {k: v.detach().clone() for k, v in evalm.state_dict().items()}
        else:
            bad += 1
            if EARLY_STOP and bad >= PATIENCE: break
    final = ema.model if ema is not None else model
    final.load_state_dict(best_state); final.eval()
    return final, dict(best_epoch=best_ep, step_at_best=best_step, epochs_run=ep, steps_per_epoch=steps_ep1, total_steps=steps, best_val_srcc=best)

def make_split(seed):
    rng = np.random.default_rng(seed); n = N_VIDEOS; perm = rng.permutation(n)
    n_te = int(round(TEST_FRAC * n)); te, rest = perm[:n_te], perm[n_te:]
    n_va = int(round(VAL_FRAC * len(rest))); va, tr = rest[:n_va], rest[n_va:]
    assert not (set(tr) & set(te)) and not (set(va) & set(te)) and not (set(tr) & set(va))
    return np.sort(tr), np.sort(va), np.sort(te)

def run(name, split, seed):
    cfg, hp = VARIANTS[name], hp_for(name); tr, va, te = split
    lo, hi = Y[tr].min(0), Y[tr].max(0) + 1e-8
    Ys = torch.from_numpy((Y - lo) / (hi - lo)).float().to(DEVICE)
    norm, adj = make_norm(tr), make_adj(Y[tr]).to(DEVICE)
    K = cfg["ens"]; pvs, pts, infos, t_train, t_test = [], [], [], 0.0, 0.0
    for k in range(K):
        _sync(); t0 = time.time()
        model, info = train_member(cfg, hp, tr, va, seed if K == 1 else seed * 1000 + k, lo, hi, Ys, norm, adj)
        _sync(); t_train += time.time() - t0
        pvs.append(predict(model, va, lo, hi, cfg))
        _sync(); t1 = time.time(); pts.append(predict(model, te, lo, hi, cfg)); _sync(); t_test += time.time() - t1     # the test set is touched once per member, after training
        infos.append(info); del model
    pv, pt = np.mean(pvs, 0), np.mean(pts, 0)
    np.save(PRED_DIR / f"{name}_s{seed}_val.npy", pv); np.save(PRED_DIR / f"{name}_s{seed}_test.npy", pt)
    res = all_metrics(Y[te], pt, Y[va], pv)
    res.update(val_mean_srcc=mean_srcc(Y[va], pv), best_epoch=float(np.mean([i["best_epoch"] for i in infos])), step_at_best=float(np.mean([i["step_at_best"] for i in infos])),
               epochs_run=float(np.mean([i["epochs_run"] for i in infos])), steps_per_epoch=infos[0]["steps_per_epoch"], total_steps=int(sum(i["total_steps"] for i in infos)),
               n_members=K, train_time_s=t_train, test_time_s=t_test, test_ms_per_clip=1000 * t_test / len(te), n_params=PARAMS[name], n_train=len(tr), n_val=len(va), n_test=len(te))
    return res
print("steps per epoch:", math.ceil(len(make_split(0)[0]) / BATCH), "| epochs (max):", EPOCHS, "| split sizes (train/val/test):", [len(a) for a in make_split(0)])

## Cell 7 - (optional) per-model hyper-parameter search on validation
Skip it (default `TUNE = False`) to use the shared fixed hyper-parameters.

In [ ]:
# ============ CELL 7 (OPTIONAL) — small hyper-parameter search per model, on VALIDATION only, on tuning seeds that are NOT among the evaluation seeds ============
# Leave TUNE = False to keep the shared fixed hyper-parameters (the reported default). If you turn it on, run it BEFORE Cell 8 (and delete runs.csv),
# because the tuned values must be fixed before the 20 evaluation seeds are run.
TUNE, TUNE_TRIALS, TUNE_SEEDS = False, 6, [1001, 1002]
SPACE = dict(LR=[1e-4, 2e-4, 5e-4], DROPOUT=[0.1, 0.2, 0.3], WD=[1e-2, 5e-2], D=[128, 256])
if TUNE:
    rng = np.random.default_rng(123); tuned_log = []
    for name in RUN_MODELS:
        trials = [{}] + [{k: v[int(rng.integers(len(v)))] for k, v in SPACE.items()} for _ in range(TUNE_TRIALS - 1)]
        best = (-9, {})
        for tr_i, ov in enumerate(trials):
            HP_OVERRIDE[name] = ov; vs = []
            for sd in TUNE_SEEDS:
                tr_, va_, te_ = make_split(sd); lo, hi = Y[tr_].min(0), Y[tr_].max(0) + 1e-8; Ys = torch.from_numpy((Y - lo) / (hi - lo)).float().to(DEVICE)
                _, info = train_member(VARIANTS[name], hp_for(name), tr_, va_, sd, lo, hi, Ys, make_norm(tr_), make_adj(Y[tr_]).to(DEVICE)); vs.append(info["best_val_srcc"])
            tuned_log.append(dict(model=name, trial=tr_i, val_mean6_srcc=float(np.mean(vs)), **ov)); print(name, tr_i, ov, round(float(np.mean(vs)), 4))
            if np.mean(vs) > best[0]: best = (float(np.mean(vs)), ov)
        HP_OVERRIDE[name] = best[1]
    HP_SOURCE = f"random search ({TUNE_TRIALS} trials/model, tuning seeds {TUNE_SEEDS}, selected on validation Mean(6) SRCC)"
    json.dump(HP_OVERRIDE, open(OUT / "tuned_hyperparameters.json", "w"), indent=1); pd.DataFrame(tuned_log).to_csv(OUT / "tables/tuning_log.csv", index=False)
    PARAMS = {m: count_params(m) for m in RUN_MODELS}
print("hyper-parameters:", HP_SOURCE)

## Cell 8 - RUN: 14 models x 20 seeds (resumable)
Prints, for every run, the training time, the test-inference time, the best epoch and the Overall metrics, and after each seed a table of all six SRCC values plus Overall PLCC, KRCC and RMSE for every model. Every number of every run is also written to `runs.csv`.

In [ ]:
# ============ CELL 8 — RUN: every model x every seed (resumable; one row per run is appended to runs.csv) ============
rows = pd.read_csv(RUNS_CSV).to_dict("records") if RUNS_CSV.exists() else []
done_keys = {(int(r["seed"]), r["model"]) for r in rows}
total = len(SEEDS) * len(RUN_MODELS)
print(f"{len(done_keys)} of {total} runs already done | models {RUN_MODELS}")
t_all = time.time()
def seed_table(seed):
    d = pd.DataFrame([r for r in rows if int(r["seed"]) == seed])
    if not len(d): return
    t = pd.DataFrame({"model": d["model"]})
    for s in TARGETS: t[f"SRCC {SCORE_NAME[s]}"] = d[f"{s}_srcc"].round(4)
    t["Mean6 SRCC"] = d["mean_srcc"].round(4)
    for m, nm in [("plcc_val", "PLCC"), ("krcc", "KRCC"), ("rmse_val", "RMSE")]: t[f"O {nm}"] = d[f"O_{m}"].round(4)
    t["train s"] = d["train_time_s"].round(1); t["test ms/clip"] = d["test_ms_per_clip"].round(2); t["best ep"] = d["best_epoch"].round(1)
    print(t.to_string(index=False))
for it, seed in enumerate(SEEDS):
    split = make_split(seed)
    for name in RUN_MODELS:
        if (seed, name) in done_keys: continue
        r = run(name, split, seed)
        r.update(seed=seed, model=name, protocol="random", epochs_cfg=EPOCHS, early_stop=EARLY_STOP)
        rows.append(r); pd.DataFrame(rows).to_csv(RUNS_CSV, index=False)
        print(f"[iteration {it+1}/{len(SEEDS)} | seed {seed}] {name:4s} | train {r['train_time_s']:6.1f}s | test {r['test_time_s']*1000:7.1f} ms ({r['test_ms_per_clip']:.2f} ms/clip) | "
              f"best epoch {r['best_epoch']:.0f}/{r['epochs_run']:.0f} x {r['steps_per_epoch']} steps | Overall SRCC {r['O_srcc']:.4f} PLCC {r['O_plcc_val']:.4f} KRCC {r['O_krcc']:.4f} RMSE {r['O_rmse_val']:.4f} | Mean(6) SRCC {r['mean_srcc']:.4f}")
        if DEVICE.type == "cuda": torch.cuda.empty_cache()
    print(f"\n----- seed {seed}: all models (val-fit mapping for PLCC/RMSE) -----"); seed_table(seed); print()
print(f"Finished. This session: {(time.time() - t_all) / 3600:.2f} h | runs.csv has {len(rows)} rows")

## Cell 9 - aggregate: all 20 seeds and top-10 seeds, mean / median / std, tests, timing, hyper-parameters, logistic parameters

In [ ]:
# ============ CELL 9 — aggregate: all seeds and the TOP-K seeds, every model x score x metric (mean / median / std), paired tests, timing, logistic parameters ============
runs = pd.read_csv(RUNS_CSV)
MODELS = [m for m in ALL_MODELS if m in set(runs["model"])]
n_seeds = int(runs["seed"].nunique()); K = min(TOP_K, n_seeds)
SUBS = {"ALL": f"all {n_seeds} seeds", "TOP": f"top {K} seeds (selected on validation)"}
SC = TARGETS + ["mean"]
print(f"{len(runs)} runs | models {MODELS} | {n_seeds} seeds | top-{K} selection key: validation Mean(6) SRCC of each run (test labels are NOT used to choose seeds)")

def sub(m, which, by="val"):
    d = runs[runs["model"] == m]
    if which == "TOP":
        key = "val_mean_srcc" if by == "val" else "mean_srcc"
        d = d.sort_values([key, "seed"], ascending=[False, True]).head(K)
    return d
def tbl(which, st, suffix, scores=SC, by="val"):
    return pd.DataFrame({SCORE_NAME[s]: [getattr(sub(m, which, by)[f"{s}_{suffix}"], st)() for m in MODELS] for s in scores}, index=MODELS)
LOWER = {"rmse_val", "rmse_test", "rmse_raw"}
HEADLINE = [("SRCC", "srcc"), ("PLCC", f"plcc_{PREFERRED_MAPPING}"), ("KRCC", "krcc"), ("RMSE", f"rmse_{PREFERRED_MAPPING}")]
TESTFIT = [("SRCC", "srcc"), ("PLCC", "plcc_test"), ("KRCC", "krcc"), ("RMSE", "rmse_test")]
def headline(which, st, mets=HEADLINE, scores=("O", "mean")):
    cols = {}
    for s in scores:
        for nm, sf in mets: cols[f"{SCORE_NAME[s]} {nm}"] = [getattr(sub(m, which)[f"{s}_{sf}"], st)() for m in MODELS]
    return pd.DataFrame(cols, index=MODELS)

def fnum(x, dec=4): return "n/a" if pd.isna(x) else f"{x:.{dec}f}"
def md_table(num, std=None, dec=4, lower_cols=()):
    cols = list(num.columns); best = {}
    for c in cols:
        lo = (c in lower_cols) or c.endswith("RMSE") or "RMSE" in c
        v = num[c].astype(float); best[c] = v.idxmin() if lo else v.idxmax()
    out = ["| " + " | ".join([num.index.name or "Model"] + cols) + " |", "|" + "|".join(["---"] * (len(cols) + 1)) + "|"]
    for i, row in num.iterrows():
        cells = []
        for c in cols:
            s = fnum(row[c], dec) + (f" ± {fnum(std.loc[i, c], dec)}" if std is not None else "")
            cells.append(f"**{s}**" if best[c] == i else s)
        out.append("| " + " | ".join([str(i)] + cells) + " |")
    return "\n".join(out)
def plain_md(df):
    out = ["| " + " | ".join([df.index.name or ""] + [str(c) for c in df.columns]) + " |", "|" + "|".join(["---"] * (len(df.columns) + 1)) + "|"]
    for i, r in df.iterrows(): out.append("| " + " | ".join([str(i)] + [str(v) for v in r.values]) + " |")
    return "\n".join(out)
def tex_table(num, std=None, dec=4, caption="", label="", lower_cols=()):
    cols = list(num.columns); best = {}
    for c in cols:
        lo = (c in lower_cols) or "RMSE" in c; v = num[c].astype(float); best[c] = v.idxmin() if lo else v.idxmax()
    L = ["\\begin{table}[t]", "\\centering", f"\\caption{{{caption}}}", f"\\label{{{label}}}", "\\begin{tabular}{l" + "c" * len(cols) + "}", "\\toprule",
         "Model & " + " & ".join(c.replace("%", "\\%").replace("_", "\\_") for c in cols) + " \\\\", "\\midrule"]
    for i, r in num.iterrows():
        cells = []
        for c in cols:
            s = fnum(r[c], dec) + (f"$\\pm${fnum(std.loc[i, c], dec)}" if std is not None else "")
            cells.append(f"\\textbf{{{s}}}" if best[c] == i else s)
        L.append(f"{i} & " + " & ".join(cells) + " \\\\")
    return "\n".join(L + ["\\bottomrule", "\\end{tabular}", "\\end{table}"])

TABLES = {}          # name -> numeric DataFrame (saved to csv / xlsx)
for which, wl in SUBS.items():
    for st in ["mean", "median", "std"]:
        TABLES[f"headline_{which}_{st}"] = headline(which, st)
        TABLES[f"headline_testfit_{which}_{st}"] = headline(which, st, TESTFIT)
        for nm, sf in HEADLINE + [("PLCC testfit", "plcc_test"), ("RMSE testfit", "rmse_test"), ("PLCC raw", "plcc_raw"), ("RMSE raw", "rmse_raw")]:
            TABLES[f"perscore_{nm.replace(' ', '_')}_{which}_{st}"] = tbl(which, st, sf)

# long-format dump: subset x model x score x metric x mapping -> mean / median / std / min / max
long = []
for which in SUBS:
    for m in MODELS:
        d = sub(m, which)
        for s in SC:
            for sf in ["srcc", "krcc", "plcc_raw", "plcc_val", "plcc_test", "rmse_raw", "rmse_val", "rmse_test"]:
                v = d[f"{s}_{sf}"]; long.append(dict(subset=SUBS[which], model=m, score=SCORE_NAME[s], metric=sf, mean=v.mean(), median=v.median(), std=v.std(), min=v.min(), max=v.max(), n=len(v)))
LONG = pd.DataFrame(long); LONG.to_csv(OUT / "tables/full_results_long.csv", index=False)

# per-seed tables: every model x seed x score x {SRCC, PLCC raw/val/test, KRCC, RMSE raw/val/test}
ps = []
for _, r in runs.iterrows():
    for s in SC:
        ps.append(dict(model=r["model"], seed=int(r["seed"]), score=SCORE_NAME[s], SRCC=r[f"{s}_srcc"], PLCC_valfit=r[f"{s}_plcc_val"], PLCC_testfit=r[f"{s}_plcc_test"], PLCC_raw=r[f"{s}_plcc_raw"],
                       KRCC=r[f"{s}_krcc"], RMSE_valfit=r[f"{s}_rmse_val"], RMSE_testfit=r[f"{s}_rmse_test"], RMSE_raw=r[f"{s}_rmse_raw"], val_mean_srcc=r["val_mean_srcc"], train_time_s=r["train_time_s"], test_time_s=r["test_time_s"]))
PER_SEED = pd.DataFrame(ps).sort_values(["model", "seed"]); PER_SEED.to_csv(OUT / "tables/per_seed_all_scores.csv", index=False)

# which seeds were selected as the TOP-K
sel = pd.DataFrame({m: sub(m, "TOP")["seed"].tolist() + [None] * (K - len(sub(m, "TOP"))) for m in MODELS}); sel.index = [f"rank {i+1}" for i in range(K)]
sel.to_csv(OUT / "tables/top_seeds_selected_on_validation.csv")
selT = pd.DataFrame({m: sub(m, "TOP", by="test")["seed"].tolist() for m in MODELS}); selT.index = [f"rank {i+1}" for i in range(K)]

# -------- DARNet side-by-side --------
def darnet_cmp(m, which, mets=TESTFIT, st="mean"):
    d = sub(m, which); rows = []
    for s in SC:
        row = {"Score": SCORE_NAME[s]}
        for i, (nm, sf) in enumerate(mets):
            v = getattr(d[f"{s}_{sf}"], st)(); row[f"{nm} DARNet"] = f"{DARNET[s][i]:.3f}"; row[f"{nm} {m}"] = f"{v:.3f}"; row[f"{nm} gap"] = f"{v - DARNET[s][i]:+.3f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("Score")

# -------- best model (chosen WITHOUT test labels) --------
val_rank = pd.Series({m: runs[runs["model"] == m]["val_mean_srcc"].mean() for m in MODELS}).sort_values(ascending=False)
BEST = val_rank.index[0]
test_rank = pd.Series({m: runs[runs["model"] == m]["mean_srcc"].mean() for m in MODELS}).sort_values(ascending=False)
overall_rank = pd.Series({m: runs[runs["model"] == m]["O_srcc"].mean() for m in MODELS}).sort_values(ascending=False)
print("Best model by mean VALIDATION Mean(6) SRCC (selection rule, no test labels):", BEST, round(val_rank[BEST], 4))
print("For reference - best by mean TEST Mean(6) SRCC:", test_rank.index[0], round(test_rank.iloc[0], 4), "| best by mean TEST Overall SRCC:", overall_rank.index[0], round(overall_rank.iloc[0], 4))

# -------- cells won (6 scores x 4 metrics = 24 cells) --------
def cells_won(which, st):
    cnt = {m: 0 for m in MODELS}
    for s in TARGETS:
        for nm, sf in HEADLINE:
            v = pd.Series({m: getattr(sub(m, which)[f"{s}_{sf}"], st)() for m in MODELS}); cnt[v.idxmin() if nm == "RMSE" else v.idxmax()] += 1
    return cnt
WON = pd.DataFrame({f"{SUBS[w]} - {st}": cells_won(w, st) for w in SUBS for st in ["mean", "median"]}).T

# -------- paired tests across the SAME seeds / splits --------
PAIRS = [("M1", "M0"), ("M2", "M0"), ("M3", "M0"), ("M4", "M3"), ("M5", "M4"), ("M6", "M5"), ("M7", "M5"), ("M8", "M5"), ("M9", "M8"), ("M10", "M9"), ("M11", "M10"), ("M12", "M11"), ("M13", "M12"),
         ("M5", "M0"), ("M13", "M5"), ("M13", "M0")]
PAIRS = [(a, b) for a, b in PAIRS if a in MODELS and b in MODELS]
def holm(p):
    o = np.argsort(p); adj = np.empty(len(p)); run_max = 0
    for rank, i in enumerate(o): run_max = max(run_max, (len(p) - rank) * p[i]); adj[i] = min(1.0, run_max)
    return adj
trows = []
for a, b in PAIRS:
    A, Bd = runs[runs["model"] == a].set_index("seed"), runs[runs["model"] == b].set_index("seed"); common = A.index.intersection(Bd.index)
    for nm, col in [("Overall SRCC", "O_srcc"), ("Mean(6) SRCC", "mean_srcc"), ("Overall PLCC (val-fit)", "O_plcc_val"), ("Overall RMSE (val-fit)", "O_rmse_val"), ("Mean(6) RMSE (val-fit)", "mean_rmse_val")]:
        x, y = A.loc[common, col].values, Bd.loc[common, col].values; diff = x - y
        try: pw = float(wilcoxon(x, y).pvalue)
        except Exception: pw = 1.0
        pt = float(ttest_rel(x, y).pvalue) if len(common) > 2 and np.std(diff) > 0 else 1.0
        trows.append(dict(model=a, vs=b, metric=nm, mean_diff=diff.mean(), seeds_better=int((diff < 0).sum() if "RMSE" in nm else (diff > 0).sum()), n=len(common), p_wilcoxon=pw, p_ttest=pt))
PAIRED = pd.DataFrame(trows)
if len(PAIRED): PAIRED["p_wilcoxon_holm"] = holm(PAIRED["p_wilcoxon"].values)
PAIRED.to_csv(OUT / "tables/paired_tests.csv", index=False)

# -------- vs DARNet (one-sample: is our mean different from DARNet's reported mean?) --------
vrows = []
for m in MODELS:
    d = runs[runs["model"] == m]
    for nm, col, ref in [("Overall SRCC", "O_srcc", DARNET["O"][0]), ("Mean(6) SRCC", "mean_srcc", DARNET["mean"][0])]:
        v = d[col].values; t = ttest_1samp(v, ref) if len(v) > 2 else None
        vrows.append(dict(model=m, metric=nm, darnet=ref, ours_mean=v.mean(), diff=v.mean() - ref, seeds_ge_darnet=int((v >= ref).sum()), n=len(v), p_ttest=float(t.pvalue) if t is not None else np.nan))
VS_DARNET = pd.DataFrame(vrows); VS_DARNET.to_csv(OUT / "tables/vs_darnet_one_sample.csv", index=False)

# -------- timing --------
TIMING = pd.DataFrame({m: dict(params=PARAMS.get(m, np.nan), members=int(runs[runs["model"] == m]["n_members"].iloc[0]), epochs_run=runs[runs["model"] == m]["epochs_run"].mean(),
                               total_steps=runs[runs["model"] == m]["total_steps"].mean(), best_epoch=runs[runs["model"] == m]["best_epoch"].mean(),
                               train_s_mean=runs[runs["model"] == m]["train_time_s"].mean(), train_s_std=runs[runs["model"] == m]["train_time_s"].std(),
                               test_s_mean=runs[runs["model"] == m]["test_time_s"].mean(), test_ms_per_clip=runs[runs["model"] == m]["test_ms_per_clip"].mean(),
                               total_train_h=runs[runs["model"] == m]["train_time_s"].sum() / 3600) for m in MODELS}).T
TIMING.index.name = "model"; TIMING.to_csv(OUT / "tables/timing_and_params.csv")
EXT = json.load(open(OUT / "feature_extraction_times.json")) if (OUT / "feature_extraction_times.json").exists() else {}

# -------- hyper-parameters per model --------
def feat_list(c):
    f = ["ResNet-50 L4"]
    if c["ms"]: f.append("ResNet L2+L3")
    if c["clip"]: f.append(CLIP_NAME or "CLIP")
    if c["ph"]: f.append("photometric")
    if c["mo"]: f.append("flow" + (" + Video-Swin" if "sw" in FEAT_AVAILABLE else ""))
    return " + ".join(f)
hrows = []
for m in MODELS:
    c, hp = VARIANTS[m], hp_for(m)
    hrows.append({"model": m, "description": DESCRIPTION[m], "input features": feat_list(c), "trainable params": PARAMS.get(m), "ensemble members": c["ens"],
                  "D": hp["D"], "dropout": hp["DROPOUT"], "heads": hp["HEADS"], "spatial layers": hp["SP_LAYERS"] if c["use_patches"] else "-", "temporal layers": hp["T_LAYERS"] if c["temporal"] == "transformer" else ("BiLSTM" if c["temporal"] == "lstm" else "-"),
                  "lr": hp["LR"], "weight decay": hp["WD"], "warm-up epochs": hp["WARMUP"], "grad clip": hp["GRAD_CLIP"], "batch": BATCH, "max epochs": EPOCHS, "early-stop patience": PATIENCE if EARLY_STOP else "off",
                  "loss": ("Huber(%.2f)+rank+PLCC" % HUBER_DELTA) if c["robust"] else "MSE+rank" + ("+consistency" if c["temp_loss"] else ""),
                  "lambda_rank": hp["LAMBDA_RANK"], "margin": hp["RANK_MARGIN"], "EMA": EMA_DECAY if c["ema"] else "-", "augmentation": f"token-drop {TOKEN_DROP}, frame-swap {FRAME_SWAP_P}" if c["aug"] else "-"})
HYPER = pd.DataFrame(hrows).set_index("model"); HYPER.to_csv(OUT / "tables/hyperparameters_per_model.csv")

# -------- finalised logistic-mapping parameters --------
lrows = []
for m in MODELS:
    d = runs[runs["model"] == m]
    for s in TARGETS:
        row = {"model": m, "score": SCORE_NAME[s]}
        for tag, tl in [("v", "val-fit"), ("t", "test-fit")]:
            for q in range(1, 5):
                v = d[f"{s}_{tag}b{q}"].dropna()
                row[f"{tl} beta{q} median"], row[f"{tl} beta{q} mean"], row[f"{tl} beta{q} std"] = v.median(), v.mean(), v.std()
        row["val-fit success rate"] = float(d[f"{s}_fitok_val"].mean()); row["test-fit success rate"] = float(d[f"{s}_fitok_test"].mean())
        lrows.append(row)
LOGI = pd.DataFrame(lrows); LOGI.to_csv(OUT / "tables/logistic_parameters.csv", index=False)

# -------- save everything numeric to one Excel workbook --------
try:
    with pd.ExcelWriter(OUT / "tables/all_results.xlsx") as xw:
        for k, v in TABLES.items(): v.to_excel(xw, sheet_name=k[:31])
        for m in MODELS:
            sub_df = PER_SEED[PER_SEED["model"] == m]; sub_df.to_excel(xw, sheet_name=f"seeds_{m}"[:31], index=False)
        PAIRED.to_excel(xw, sheet_name="paired_tests", index=False); VS_DARNET.to_excel(xw, sheet_name="vs_darnet", index=False)
        TIMING.to_excel(xw, sheet_name="timing_params"); HYPER.to_excel(xw, sheet_name="hyperparameters"); LOGI.to_excel(xw, sheet_name="logistic_params", index=False); WON.to_excel(xw, sheet_name="cells_won")
except Exception as e:
    print("Excel export skipped:", type(e).__name__, e)
for k, v in TABLES.items(): v.to_csv(OUT / f"tables/{k}.csv")

# -------- print the main results --------
def show(title, txt): print("\n" + "=" * 8, title, "=" * 8 + "\n" + txt)
for which, wl in SUBS.items():
    for st in ["mean", "median"]:
        show(f"Overall and Mean(6): SRCC / PLCC / KRCC / RMSE - {wl} - {st} (PLCC, RMSE: {PREFERRED_MAPPING}-fitted logistic mapping)", headline(which, st).round(4).to_string())
show("Per-score SRCC (mean), " + SUBS["ALL"], tbl("ALL", "mean", "srcc").round(4).to_string())
show("Cells won out of 24 (6 scores x SRCC, PLCC, KRCC, RMSE)", WON.to_string())
show("Training / test time and size", TIMING.round(3).to_string())

## Cell 10 - figures

In [ ]:
# ============ CELL 10 — publication figures (PNG 300 dpi + PDF) ============
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#2b2b2a", "#8a8984", "#e4e3df"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "figure.dpi": 110, "savefig.bbox": "tight", "legend.frameon": False})
FIGS = []
def savefig(fig, name, title):
    fig.savefig(OUT / "figures" / f"{name}.png", dpi=300); fig.savefig(OUT / "figures" / f"{name}.pdf"); FIGS.append((name, title))
    if "ipykernel" in sys.modules: plt.show()
    plt.close(fig)
def col(m): return ORANGE if m == BEST else BLUE
x = np.arange(len(MODELS)); lab = MODELS

def dot_panel(ax, suffix, score, ylabel, ref=None, lower=False):
    mu = np.array([runs[runs["model"] == m][f"{score}_{suffix}"].mean() for m in MODELS]); sd = np.array([runs[runs["model"] == m][f"{score}_{suffix}"].std() for m in MODELS])
    top = np.array([sub(m, "TOP")[f"{score}_{suffix}"].mean() for m in MODELS])
    for i, m in enumerate(MODELS): ax.errorbar(i, mu[i], yerr=sd[i], fmt="o", color=col(m), capsize=3, ms=6, lw=1.4)
    ax.plot(x, top, "D", color=INK, ms=4, label=f"mean of top {K} seeds")
    ax.plot([], [], "o", color=BLUE, label=f"mean ± sd, all {n_seeds} seeds")
    if ref is not None: ax.axhline(ref, color=INK, ls="--", lw=1); ax.annotate(f"DARNet {ref:.3f}", (len(MODELS) - 0.6, ref), ha="right", va="bottom" if not lower else "top", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_ylabel(ylabel)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
dot_panel(ax[0], "srcc", "O", "SRCC (Overall)", DARNET["O"][0]); dot_panel(ax[1], "srcc", "mean", "SRCC (mean of the six scores)", DARNET["mean"][0]); ax[0].legend(loc="lower left", fontsize=8)
ax[0].set_title("Overall quality score", loc="left"); ax[1].set_title("Mean of TSV, B, SR, S, U, Overall", loc="left")
savefig(fig, "fig1_srcc_overall_and_mean6", "SRCC of every model (mean ± sd over seeds, diamonds: top-K seeds) against DARNet")

fig, ax = plt.subplots(2, 2, figsize=(11, 6.4))
for a, (suf, sc, yl, ref, lo) in zip(ax.ravel(), [("srcc", "O", "Overall SRCC", DARNET["O"][0], False), (f"plcc_{PREFERRED_MAPPING}", "O", f"Overall PLCC ({PREFERRED_MAPPING}-fit)", DARNET["O"][1], False),
                                              ("krcc", "O", "Overall KRCC", DARNET["O"][2], False), (f"rmse_{PREFERRED_MAPPING}", "O", f"Overall RMSE ({PREFERRED_MAPPING}-fit, lower is better)", DARNET["O"][3], True)]):
    dot_panel(a, suf, sc, yl, ref, lo)
ax[0, 0].legend(loc="lower left", fontsize=7)
savefig(fig, "fig2_four_metrics_overall", "Overall SRCC, PLCC, KRCC and RMSE of every model against DARNet")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for a, (sc, yl, ref) in zip(ax, [("O", "Overall SRCC", DARNET["O"][0]), ("mean", "Mean(6) SRCC", DARNET["mean"][0])]):
    data = [runs[runs["model"] == m][f"{sc}_srcc"].values for m in MODELS]
    bp = a.boxplot(data, positions=x, widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color=INK, lw=1.4))
    for p_, m in zip(bp["boxes"], MODELS): p_.set(facecolor=col(m), alpha=0.35, edgecolor=col(m))
    rng = np.random.default_rng(0)
    for i, d_ in enumerate(data): a.plot(i + rng.uniform(-0.15, 0.15, len(d_)), d_, ".", color=INK, ms=3, alpha=0.6)
    a.axhline(ref, color=INK, ls="--", lw=1); a.annotate(f"DARNet {ref:.3f}", (len(MODELS) - 0.6, ref), ha="right", va="bottom", fontsize=8)
    a.set_xticks(x); a.set_xticklabels(lab); a.set_ylabel(yl)
savefig(fig, "fig3_boxplots_srcc", f"Distribution of test SRCC over the {n_seeds} random splits (each dot = one iteration)")

tab = tbl("ALL", "mean", "srcc"); tab.loc["DARNet"] = [DARNET[s][0] for s in SC]; tab = tab.loc[["DARNet"] + MODELS]
fig, ax = plt.subplots(1, 2, figsize=(11, 0.42 * len(tab) + 1.6), gridspec_kw=dict(width_ratios=[1, 1]))
im = ax[0].imshow(tab.values, cmap="Blues", aspect="auto", vmin=np.nanmin(tab.values) - 0.02, vmax=np.nanmax(tab.values))
ax[0].set_xticks(range(tab.shape[1])); ax[0].set_xticklabels(tab.columns); ax[0].set_yticks(range(len(tab))); ax[0].set_yticklabels(tab.index); ax[0].grid(False)
for i in range(tab.shape[0]):
    for j in range(tab.shape[1]): ax[0].text(j, i, f"{tab.values[i, j]:.3f}", ha="center", va="center", fontsize=7.5, color="white" if tab.values[i, j] > np.nanmin(tab.values) + 0.6 * (np.nanmax(tab.values) - np.nanmin(tab.values)) else INK)
ax[0].set_title(f"Mean SRCC per score ({n_seeds} seeds)", loc="left")
gap = tab.iloc[1:].values - tab.loc["DARNet"].values[None]; lim = max(0.02, np.nanmax(np.abs(gap)))
ax[1].imshow(gap, cmap="RdBu", aspect="auto", vmin=-lim, vmax=lim); ax[1].set_xticks(range(tab.shape[1])); ax[1].set_xticklabels(tab.columns); ax[1].set_yticks(range(len(gap))); ax[1].set_yticklabels(MODELS); ax[1].grid(False)
for i in range(gap.shape[0]):
    for j in range(gap.shape[1]): ax[1].text(j, i, f"{gap[i, j]:+.3f}", ha="center", va="center", fontsize=7.5, color="white" if abs(gap[i, j]) > 0.6 * lim else INK)
ax[1].set_title("Gap to DARNet (blue = above DARNet, red = below)", loc="left")
savefig(fig, "fig4_heatmap_per_score_srcc", "Mean SRCC per score and the gap to DARNet")

fig, ax = plt.subplots(2, 2, figsize=(11, 6.2))
cmp_models = [m for m in ["M0", BEST] if m in MODELS]; cmp_models = list(dict.fromkeys(cmp_models))
for a, (i, nm, sf) in zip(ax.ravel(), [(0, "SRCC", "srcc"), (1, "PLCC", f"plcc_test"), (2, "KRCC", "krcc"), (3, "RMSE (lower is better)", "rmse_test")]):
    w = 0.8 / (1 + len(cmp_models)); xs = np.arange(len(SC))
    a.bar(xs - 0.4 + w / 2, [DARNET[s][i] for s in SC], w, color=INK, alpha=0.75, label="DARNet (paper)")
    for q, m in enumerate(cmp_models):
        a.bar(xs - 0.4 + w * (q + 1.5), [runs[runs["model"] == m][f"{s}_{sf}"].mean() for s in SC], w, color=ORANGE if m == BEST else BLUE, alpha=1.0 if m == BEST else 0.55, label=f"{m}" + (" (best)" if m == BEST else ""))
    a.set_xticks(xs); a.set_xticklabels([SCORE_NAME[s] for s in SC]); a.set_ylabel(nm)
    if i < 3: a.set_ylim(0.6, 1.0)
ax[0, 0].legend(ncol=3, fontsize=8, loc="lower left")
savefig(fig, "fig5_per_score_vs_darnet", "Per-score comparison of DARNet, M0 and the selected best model (PLCC and RMSE with test-fitted mapping, as in the paper)")

fig, ax = plt.subplots(figsize=(9, 3.6)); xs = np.arange(len(MODELS))
mu = np.array([runs[runs["model"] == m]["mean_srcc"].mean() for m in MODELS]); se = np.array([runs[runs["model"] == m]["mean_srcc"].sem() for m in MODELS])
ci = se * tdist.ppf(0.975, max(n_seeds - 1, 1))
ax.plot(xs, mu, "-", color=MUTED, lw=1); ax.errorbar(xs, mu, yerr=ci, fmt="o", color=BLUE, capsize=3); ax.axhline(DARNET["mean"][0], color=INK, ls="--", lw=1)
ax.annotate(f"DARNet {DARNET['mean'][0]:.3f}", (len(MODELS) - 0.6, DARNET["mean"][0]), ha="right", va="bottom", fontsize=8)
for i, m in enumerate(MODELS): ax.annotate(m, (i, mu[i] + ci[i]), ha="center", va="bottom", fontsize=7.5, xytext=(0, 3), textcoords="offset points")
ax.set_xticks(xs); ax.set_xticklabels(MODELS); ax.set_ylabel("Mean(6) SRCC (95% CI)")
savefig(fig, "fig6_ablation_ladder", "Ablation ladder: effect of each added component on Mean(6) SRCC")

fig, ax = plt.subplots(figsize=(7, 4.4))
tt = TIMING.loc[MODELS]; ys = np.array([runs[runs["model"] == m]["mean_srcc"].mean() for m in MODELS])
ax.scatter(tt["train_s_mean"], ys, s=20 + 160 * (tt["params"].astype(float) / tt["params"].astype(float).max()), c=[col(m) for m in MODELS], alpha=0.8, edgecolor=INK, lw=0.5)
for m, xx, yy in zip(MODELS, tt["train_s_mean"], ys): ax.annotate(m, (xx, yy), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("mean training time per iteration (s, log scale)"); ax.set_ylabel("Mean(6) SRCC"); ax.set_title("Accuracy vs training cost (bubble size = trainable parameters)", loc="left")
savefig(fig, "fig7_cost_vs_accuracy", "Training time and size against accuracy")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, (c_, yl) in zip(ax, [("best_epoch", "epoch with best validation SRCC"), ("train_time_s", "training time per iteration (s)")]):
    bp = a.boxplot([runs[runs["model"] == m][c_].values for m in MODELS], positions=x, widths=0.55, patch_artist=True, showfliers=False, medianprops=dict(color=INK))
    for p_, m in zip(bp["boxes"], MODELS): p_.set(facecolor=col(m), alpha=0.35, edgecolor=col(m))
    a.set_xticks(x); a.set_xticklabels(lab); a.set_ylabel(yl)
ax[1].set_yscale("log")
savefig(fig, "fig8_best_epoch_and_time", "Selected epoch and training time per iteration")

# scatter of predictions vs MOS for the selected model, on its median-Overall-SRCC seed
dB = runs[runs["model"] == BEST].sort_values("O_srcc"); seed_med = int(dB.iloc[len(dB) // 2]["seed"])
tr_, va_, te_ = make_split(seed_med); pv = np.load(PRED_DIR / f"{BEST}_s{seed_med}_val.npy"); pt = np.load(PRED_DIR / f"{BEST}_s{seed_med}_test.npy"); j = TARGETS.index("O")
fv, _, _ = fit_mapping(Y[va_][:, j], pv[:, j]); ft, _, _ = fit_mapping(Y[te_][:, j], pt[:, j])
fig, ax = plt.subplots(1, 3, figsize=(11, 3.7), sharex=True, sharey=True)
for a, (ttl, yy) in zip(ax, [("raw prediction", pt[:, j]), ("mapping fitted on validation", apply_mapping(fv, pt[:, j])), ("mapping fitted on test", apply_mapping(ft, pt[:, j]))]):
    a.scatter(Y[te_][:, j], yy, s=10, color=BLUE, alpha=0.7); lo_, hi_ = Y[:, j].min(), Y[:, j].max(); a.plot([lo_, hi_], [lo_, hi_], color=INK, ls="--", lw=1)
    a.set_title(f"{ttl}\nSRCC {_corr(spearmanr, Y[te_][:, j], yy):.3f}  PLCC {_corr(pearsonr, Y[te_][:, j], yy):.3f}  RMSE {_rmse(Y[te_][:, j], yy):.3f}", loc="left", fontsize=8.5); a.set_xlabel("subjective MOS (Overall)")
ax[0].set_ylabel(f"{BEST} prediction (seed {seed_med})")
savefig(fig, "fig9_prediction_vs_mos", f"Predicted vs subjective Overall MOS for {BEST} (median-performing seed {seed_med})")

fig, ax = plt.subplots(2, 3, figsize=(11, 5.6))
dBest = runs[runs["model"] == BEST]
for a, s in zip(ax.ravel(), TARGETS):
    jj = TARGETS.index(s); allp = np.concatenate([np.load(PRED_DIR / f"{BEST}_s{int(sd)}_test.npy")[:, jj] for sd in dBest["seed"]]); gx = np.linspace(allp.min(), allp.max(), 200)
    for _, r in dBest.iterrows():
        b_ = [r[f"{s}_vb{q}"] for q in range(1, 5)]
        if not np.any(np.isnan(b_)): a.plot(gx, NonLinearFit.four_param_logistic_QA(gx, *b_), color=BLUE, alpha=0.25, lw=1)
    bm = [dBest[f"{s}_vb{q}"].median() for q in range(1, 5)]
    if not np.any(np.isnan(bm)): a.plot(gx, NonLinearFit.four_param_logistic_QA(gx, *bm), color=ORANGE, lw=2, label="median parameters")
    a.plot(gx, gx, color=INK, ls="--", lw=1, label="identity"); a.set_title(SCORE_NAME[s], loc="left"); a.set_xlabel("raw prediction"); a.set_ylabel("mapped")
ax[0, 0].legend(fontsize=7)
savefig(fig, "fig10_logistic_mapping_curves", f"Validation-fitted 4-parameter logistic mappings of {BEST} (thin: individual seeds, thick: median parameters)")

fig, ax = plt.subplots(figsize=(7.5, 0.34 * len(PAIRS) + 1.4)); yl_ = []
for i, (a_, b_) in enumerate(PAIRS):
    A_, B_ = runs[runs["model"] == a_].set_index("seed"), runs[runs["model"] == b_].set_index("seed"); cm = A_.index.intersection(B_.index); dd = (A_.loc[cm, "mean_srcc"] - B_.loc[cm, "mean_srcc"]).values
    ci_ = (tdist.ppf(0.975, max(len(dd) - 1, 1)) * dd.std(ddof=1) / np.sqrt(len(dd))) if len(dd) > 1 else 0
    pr = PAIRED[(PAIRED["model"] == a_) & (PAIRED["vs"] == b_) & (PAIRED["metric"] == "Mean(6) SRCC")]
    sig = len(pr) and pr["p_wilcoxon"].iloc[0] < 0.05
    ax.errorbar(dd.mean(), i, xerr=ci_, fmt="o", color=ORANGE if sig else BLUE, capsize=3); yl_.append(f"{a_} vs {b_}")
ax.axvline(0, color=INK, lw=1); ax.set_yticks(range(len(PAIRS))); ax.set_yticklabels(yl_); ax.invert_yaxis(); ax.set_xlabel("difference in Mean(6) SRCC (paired over seeds, 95% CI); orange: Wilcoxon p < 0.05 (uncorrected)")
savefig(fig, "fig11_paired_differences", "Paired differences between models on the same splits")
print(f"{len(FIGS)} figures saved to {OUT/'figures'}")

## Cell 11 - README with every table (and LaTeX tables)

In [ ]:
# ============ CELL 11 — write README_results.md (+ LaTeX tables) with every table, ready for the paper ============
def f3(x): return f"{x:.3f}"
def head_md(which, st, mets=HEADLINE, with_std=True):
    num = headline(which, st, mets)
    return md_table(num, std=headline(which, "std", mets) if (with_std and st == "mean") else None)
def seedlist(m, which="TOP"): return ", ".join(str(int(s)) for s in sub(m, which)["seed"])
bestA = runs[runs["model"] == BEST]; bestT = sub(BEST, "TOP")
d_o = DARNET["O"]; d_m = DARNET["mean"]
rows_vs = VS_DARNET.pivot(index="model", columns="metric", values="diff").loc[MODELS]
R = []
R.append(f"""# CV-STQA on MAC-VQA - M0 to M{len(MODELS)-1 if len(MODELS) else 0} ({n_seeds} random splits each, all 1,000 clips)

Everything in this folder was produced by one run of `CVSTQA_M0-M13_20seeds.ipynb`. Tables are in `tables/`, figures in `figures/`, per-run raw numbers in `runs.csv`.

## 1. Summary
* **Selected model: {BEST}** ({DESCRIPTION[BEST]}). It was chosen by the highest *validation* Mean(6) SRCC averaged over the {n_seeds} iterations (test labels play no role in the choice).
* {BEST}: Overall SRCC **{bestA['O_srcc'].mean():.4f}** (all {n_seeds}) / **{bestT['O_srcc'].mean():.4f}** (top {K}); Mean(6) SRCC **{bestA['mean_srcc'].mean():.4f}** / **{bestT['mean_srcc'].mean():.4f}**.
  DARNet (paper): Overall SRCC {d_o[0]:.3f}, Mean(6) SRCC {d_m[0]:.3f}.
* Overall RMSE ({PREFERRED_MAPPING}-fit mapping) {bestA['O_rmse_'+PREFERRED_MAPPING].mean():.4f}; with the paper's test-fitted mapping {bestA['O_rmse_test'].mean():.4f} (DARNet {d_o[3]:.3f}).
* Seeds beating DARNet's Overall SRCC ({d_o[0]:.3f}): {int((bestA['O_srcc'] >= d_o[0]).sum())} of {len(bestA)}; Mean(6) SRCC ({d_m[0]:.3f}): {int((bestA['mean_srcc'] >= d_m[0]).sum())} of {len(bestA)}.
* For reference, the model with the highest mean *test* Mean(6) SRCC is {test_rank.index[0]} ({test_rank.iloc[0]:.4f}); with the highest mean test Overall SRCC {overall_rank.index[0]} ({overall_rank.iloc[0]:.4f}).

## 2. Protocol
| Item | Value |
|---|---|
| Dataset | MAC-VQA, **{N_VIDEOS} clips** (all used), six MOS labels: TSV, B, SR, S, U, Overall |
| Iterations (random seeds / splits) | **{n_seeds}** per model; the same {n_seeds} splits for every model |
| Models | {len(MODELS)}: {', '.join(MODELS)} |
| Training runs | {len(runs)} |
| Split per iteration | random {runs['n_train'].iloc[0]} train / {runs['n_val'].iloc[0]} validation / {runs['n_test'].iloc[0]} test |
| Epochs | up to **{EPOCHS}**, early stopping {'on (patience %d)' % PATIENCE if EARLY_STOP else 'off'}; the best-validation epoch is kept |
| Steps (batches) per epoch | **{int(runs['steps_per_epoch'].iloc[0])}** (batch {BATCH}) |
| Test evaluation | once per run, after training |
| Metrics | SRCC, PLCC, KRCC, RMSE for TSV, B, SR, S, U, Overall (O) and their mean, Mean(6); mean, median and std over the iterations |
| Logistic mapping | 4-parameter logistic of `nithincbabu7/non-lin-fit-python` (`four_param_logistic_QA`, `curve_fit`); **fitted on validation predictions and applied fixed to test** (headline) and fitted on test predictions (paper protocol, reference). SRCC and KRCC do not depend on it |
| Top-{K} subset | for every model the {K} best of the {n_seeds} iterations, ranked by *validation* Mean(6) SRCC |
| Hyper-parameters | {HP_SOURCE} |
| Features | ResNet-50 (L4 + L2/L3), {CLIP_NAME}, photometric descriptors, optical-flow descriptors{', Video-Swin-T' if 'sw' in FEAT_AVAILABLE else ''}; frozen and cached once |

> On top-{K}: choosing the best {K} of {n_seeds} iterations is a selection step. Here it uses validation scores only, so the reported top-{K} numbers do not use test labels to pick seeds, but they are still a best-of subset and should be reported *next to* the all-{n_seeds} numbers (both are given everywhere). A top-{K} ranking by test score is listed in section 11 for reference only and must not be used in a paper.

## 3. Models
""")
mt = pd.DataFrame({"description": [DESCRIPTION[m] for m in MODELS], "input features": [feat_list(VARIANTS[m]) for m in MODELS], "trainable params": [f"{PARAMS[m]:,}" for m in MODELS],
                   "members": [VARIANTS[m]["ens"] for m in MODELS]}, index=MODELS); mt.index.name = "Model"
R.append(plain_md(mt))
R.append("\n## 4. Main results (Overall and the six-score mean)\nPLCC and RMSE use the validation-fitted logistic mapping. Bold = best in column (lowest for RMSE).\n")
for which in SUBS:
    for st in ["mean", "median"]:
        R.append(f"### 4.{'1' if which=='ALL' else '2'}{'a' if st=='mean' else 'b'} {SUBS[which]} - {st}" + (" ± std" if st == "mean" else "") + "\n" + head_md(which, st) + "\n")
R.append("### 4.3 Same, with the mapping fitted on the test predictions (the paper's protocol; optimistic, for comparison with DARNet only)\n")
for which in SUBS: R.append(f"**{SUBS[which]} - mean**\n\n" + head_md(which, "mean", TESTFIT) + "\n")
R.append("## 5. Per-score results\n")
for which in SUBS:
    for st in ["mean", "median"]:
        for nm, sf in HEADLINE:
            R.append(f"### 5 - {nm}, {SUBS[which]}, {st}\n" + md_table(tbl(which, st, sf), std=tbl(which, "std", sf) if st == "mean" else None) + "\n")
R.append("## 6. Comparison with DARNet (paper, Tables I-II)\n### 6.1 DARNet\n")
dn = pd.DataFrame({SCORE_NAME[s]: DARNET[s] for s in SC}, index=["SRCC", "PLCC", "KRCC", "RMSE"]).T; dn.index.name = "Score"; R.append(plain_md(dn.map(f3) if hasattr(dn, "map") else dn.applymap(f3)))
R.append(f"\n### 6.2 Selected model {BEST} vs DARNet, all {n_seeds} seeds (test-fitted mapping, as in the paper; gap = ours - DARNet, for RMSE negative is better)\n" + plain_md(darnet_cmp(BEST, "ALL")))
R.append(f"\n### 6.3 {BEST} vs DARNet, top {K} seeds\n" + plain_md(darnet_cmp(BEST, "TOP")))
R.append(f"\n### 6.4 {BEST} vs DARNet with the stricter validation-fitted mapping, all {n_seeds} seeds\n" + plain_md(darnet_cmp(BEST, "ALL", HEADLINE)))
R.append("\n### 6.5 Every model against DARNet (mean over seeds, difference = ours - DARNet)\n" + plain_md(VS_DARNET.assign(**{c: VS_DARNET[c].round(4) for c in ["ours_mean", "diff", "p_ttest"]}).set_index("model")))
R.append("\nDARNet's numbers are copied from its paper (its own splits and randomness); `p_ttest` is a one-sample t-test of our per-seed values against DARNet's mean and says only whether the means differ, not that the two methods were compared on the same splits.\n")
R.append("## 7. Which model wins\n### 7.1 Cells won (24 = 6 scores x SRCC, PLCC, KRCC, RMSE)\n" + plain_md(WON) + "\n")
R.append("### 7.2 Paired tests on the same splits (Wilcoxon signed-rank, Holm-corrected across all rows; t-test uncorrected)\n" + plain_md(PAIRED.round(4).set_index("model") if len(PAIRED) else pd.DataFrame()) + "\n")
R.append("## 8. Training and test time\n")
tm = TIMING.copy(); tm["params"] = tm["params"].map(lambda v: f"{int(v):,}"); R.append(plain_md(tm.round(3)))
R.append("\n`train_s_mean`: wall-clock training time per iteration including validation after every epoch (for M13 the sum over its ensemble members). `test_s_mean` / `test_ms_per_clip`: time to predict the test split from cached features (GPU, batch 64). "
         "Feature extraction (once, all clips): " + (", ".join(f"{k} {v/60:.1f} min" for k, v in EXT.items() if k.endswith("_s")) if EXT else "not recorded in this session") + ".\n")
R.append("## 9. Hyper-parameters and logistic-mapping parameters\n### 9.1 Fixed hyper-parameters per model\n" + f"Source: {HP_SOURCE}.\n\n" + plain_md(HYPER.drop(columns=["description"]).astype(str)))
R.append(f"\n### 9.2 Finalised 4-parameter logistic mapping (median over the {n_seeds} validation fits; MOS units; f(y) = (b1-b2)/(1+exp((b3-y)/|b4|)) + b2)\nFull table with mean, std and the test-fitted version: `tables/logistic_parameters.csv`. Because the predictions are already close to linear in MOS, individual beta values can differ a lot between seeds (the logistic is then fitted on only its central part); the *curve* is what is stable - see `figures/fig10_logistic_mapping_curves.png`. The medians below are the finalised values.\n")
L = LOGI[LOGI["score"].isin(["Overall"])][["model", "score"] + [f"val-fit beta{q} median" for q in range(1, 5)] + ["val-fit success rate"]].copy()
L.columns = ["model", "score", "beta1", "beta2", "beta3", "beta4", "fit success"]; R.append("**Overall score, all models**\n\n" + plain_md(L.round(4).set_index("model")))
Lb = LOGI[LOGI["model"] == BEST][["score"] + [f"val-fit beta{q} median" for q in range(1, 5)] + [f"test-fit beta{q} median" for q in range(1, 5)]].copy()
Lb.columns = ["score", "val b1", "val b2", "val b3", "val b4", "test b1", "test b2", "test b3", "test b4"]; R.append(f"\n**{BEST}, every score (val-fit and test-fit medians)**\n\n" + plain_md(Lb.round(4).set_index("score")) + "\n")
R.append("## 10. Figures\n" + "\n".join(f"* `figures/{n}.png` - {t}" for n, t in FIGS) + "\n")
R.append(f"## 11. Seeds\n### 11.1 Top-{K} iterations per model, selected on validation Mean(6) SRCC (used for the top-{K} columns above)\n" + plain_md(sel.astype(str).replace("None", "-")) + "\n")
R.append(f"### 11.2 Reference only - top-{K} ranked by TEST Mean(6) SRCC (this is selection on the test set; do not report)\n" + plain_md(selT.astype(str)) + "\n")
R.append("""## 12. Limitations
1. **Random split.** Clips of the same patient can be in training and test (as in DARNet's protocol). A patient-independent split has not been run and would probably lower all scores.
2. **DARNet values are taken from the paper**, not re-run on these splits.
3. **Top-K subset.** It is a best-of selection (here by validation score). Report it only together with the all-seed result.
4. **Frozen backbones.** All networks train only the heads on cached features, unlike DARNet which trains its backbones.
5. **Hyper-parameters** are shared and fixed (unless the optional tuning cell was used); they were not tuned on the test set.
6. **Test-fitted logistic mapping** uses test labels and is therefore optimistic; the validation-fitted mapping is the headline.

## 13. Files
| File | Content |
|---|---|
| `runs.csv` | one row per (model, seed): all metrics, logistic parameters, timing, epochs |
| `tables/full_results_long.csv` | model x score x metric x mapping: mean, median, std, min, max, for all seeds and top-K |
| `tables/per_seed_all_scores.csv` | every model x seed x score with SRCC, PLCC, KRCC, RMSE |
| `tables/all_results.xlsx` | all tables, one sheet each, plus one sheet per model with every seed |
| `tables/paired_tests.csv`, `vs_darnet_one_sample.csv` | significance tests |
| `tables/timing_and_params.csv`, `hyperparameters_per_model.csv`, `logistic_parameters.csv` | cost, settings, mapping parameters |
| `tables/latex_tables.tex` | the main tables as LaTeX |
| `figures/*.png`, `*.pdf` | figures (300 dpi) |
| `preds/` | validation and test predictions of every run |
""")
README = "\n".join(R); (OUT / "README_results.md").write_text(README)

LT = []
for which in SUBS:
    LT.append(tex_table(headline(which, "mean"), std=headline(which, "std"), caption=f"Test results, mean $\\pm$ std over {SUBS[which]}. PLCC and RMSE use the validation-fitted logistic mapping.", label=f"tab:main_{which.lower()}", dec=3))
LT.append(tex_table(tbl("ALL", "mean", "srcc"), std=tbl("ALL", "std", "srcc"), caption=f"SRCC per score, mean $\\pm$ std over {n_seeds} seeds.", label="tab:srcc_per_score", dec=3))
LT.append(tex_table(tbl("ALL", "mean", "rmse_val"), caption="RMSE per score (validation-fitted mapping), mean over seeds.", label="tab:rmse_per_score", dec=3))
LT.append(tex_table(TIMING[["params", "train_s_mean", "test_ms_per_clip"]].astype(float).rename(columns={"params": "Params", "train_s_mean": "Train s", "test_ms_per_clip": "Test ms/clip"}), caption="Model size and cost.", label="tab:cost", dec=2))
(OUT / "tables/latex_tables.tex").write_text("\n\n".join(LT))
print(README[:3500]); print("...\nREADME_results.md written:", len(README), "characters")

## Cell 12 - zip the results

In [ ]:
# ============ CELL 12 — save everything into one zip ============
info = dict(torch=torch.__version__, python=platform.python_version(), device=str(DEVICE), gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
            seeds=SEEDS, epochs=EPOCHS, batch=BATCH, early_stop=EARLY_STOP, patience=PATIENCE, top_k=TOP_K, models=MODELS, hyperparameters=HP, hp_overrides=HP_OVERRIDE, hp_source=HP_SOURCE,
            m13_extras=dict(EMA_DECAY=EMA_DECAY, TOKEN_DROP=TOKEN_DROP, FRAME_SWAP_P=FRAME_SWAP_P, HUBER_DELTA=HUBER_DELTA, LAMBDA_PLCC=LAMBDA_PLCC, ATTR_W=ATTR_W, ENS_K=ENS_K),
            features=FEAT_AVAILABLE, clip_backbone=CLIP_NAME, n_videos=int(N_VIDEOS), preferred_mapping=PREFERRED_MAPPING, best_model_by_validation=str(BEST), darnet_reference=DARNET)
json.dump(info, open(OUT / "config_and_environment.json", "w"), indent=1, default=str)
ZIP = WORK / "cvstqa_m13_results.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file(): z.write(p, p.relative_to(WORK))
print(f"Saved {ZIP} ({ZIP.stat().st_size/1e6:.1f} MB) with {sum(1 for _ in OUT.rglob('*') if _.is_file())} files")
if "ipykernel" in sys.modules:
    try:
        from IPython.display import FileLink, display; display(FileLink(str(ZIP.relative_to(WORK))))
    except Exception: pass